# CNLA — Clean Recovery / Final ML Pipeline

This notebook is a cleaned, restart-safe version of the DGA project pipeline.

**Original dataset filenames are preserved exactly:**
- `top-1m.csv`
- `dga_data.csv`
- `dga_domains_full.csv`

The notebook intentionally excludes experiments that were already rejected:
Random Forest as the final model, VC-pattern features, rare-bigram features,
word-likeness gating, standalone word-level TF-IDF, and basic structural-feature
augmentation.

The primary detector is **character TF-IDF (2–5) + Linear SVM**.

The unseen-family evaluation keeps **bamital, goz, and matsnu** completely out
of DGA training data.


In [1]:
# 1. Install and import dependencies
!pip -q install tldextract joblib

import os
import re
import math
import time
import joblib
import numpy as np
import pandas as pd
import tldextract

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

print("Dependencies ready.")


Dependencies ready.


In [2]:
# 2. Verify the original files exist
required_files = [
    "/content/top-1m.csv",
    "/content/dga_data.csv",
    "/content/dga_domains_full.csv",
]

for f in required_files:
    print(f, "->", "FOUND" if os.path.exists(f) else "MISSING")

assert all(os.path.exists(f) for f in required_files), (
    "Upload the three original CSV files to /content before continuing."
)


/content/top-1m.csv -> FOUND
/content/dga_data.csv -> FOUND
/content/dga_domains_full.csv -> FOUND


In [3]:
# 3. Load the three original datasets
legit = pd.read_csv(
    "/content/top-1m.csv",
    header=None,
    names=["rank", "domain"]
)

old_dga = pd.read_csv(
    "/content/dga_data.csv",
    header=None,
    names=["isDGA", "domain", "host", "subclass"]
)

if str(old_dga.iloc[0]["isDGA"]).lower() == "isdga":
    old_dga = old_dga.iloc[1:].reset_index(drop=True)

# IMPORTANT: dga_domains_full.csv is comma-separated; do NOT use sep="\\t".
new_dga = pd.read_csv(
    "/content/dga_domains_full.csv",
    header=None,
    names=["label", "family", "domain"]
)

if str(new_dga.iloc[0]["label"]).lower() == "dga":
    new_dga = new_dga.iloc[1:].reset_index(drop=True)

print("top-1m:", legit.shape)
print("dga_data:", old_dga.shape)
print("dga_domains_full:", new_dga.shape)


top-1m: (102467, 2)
dga_data: (38754, 4)
dga_domains_full: (70951, 3)


In [4]:
# 4. Cleaning helpers
def clean_domain(domain):
    if pd.isna(domain):
        return ""

    domain = str(domain).strip().lower()
    domain = re.sub(r"^https?://", "", domain)
    domain = domain.split("/")[0]
    domain = domain.split(":")[0]
    domain = domain.rstrip(".")

    return domain


def is_valid_domain(domain):
    if not domain:
        return False

    return bool(re.fullmatch(
        r"[a-z0-9](?:[a-z0-9.-]*[a-z0-9])?",
        domain
    ))


# Public-Suffix-List-aware extraction used for the DGA dataset.
def extract_dga_label_v2(domain):
    domain = clean_domain(domain)

    if not domain:
        return ""

    ext = tldextract.extract(domain)

    registered_label = ext.domain
    subdomain = ext.subdomain

    if subdomain:
        for label in subdomain.split("."):
            if label:
                return label

    return registered_label


# For the top-1M legitimate source, use the registered-domain label.
# This avoids treating "www" as the DGA string for ordinary web domains.
def extract_registered_label(domain):
    domain = clean_domain(domain)

    if not domain:
        return ""

    ext = tldextract.extract(domain)
    return ext.domain


In [5]:
# 5. Clean source datasets

legit_clean = legit.copy()
legit_clean["domain"] = legit_clean["domain"].apply(clean_domain)
legit_clean = legit_clean[legit_clean["domain"].apply(is_valid_domain)].copy()
legit_clean = legit_clean.drop_duplicates("domain").reset_index(drop=True)
legit_clean["dga_string"] = legit_clean["domain"].apply(extract_registered_label)
legit_clean["label"] = 0

old_clean = old_dga.copy()
old_clean["domain"] = old_clean["domain"].apply(clean_domain)
old_clean = old_clean[
    old_clean["isDGA"].astype(str).str.lower() == "dga"
].copy()
old_clean = old_clean[old_clean["domain"].apply(is_valid_domain)].copy()
old_clean = old_clean.drop_duplicates("domain").reset_index(drop=True)
old_clean["dga_string"] = old_clean["domain"].apply(extract_dga_label_v2)
old_clean["label"] = 1

new_dga_only = new_dga.copy()
new_dga_only["domain"] = new_dga_only["domain"].apply(clean_domain)
new_dga_only = new_dga_only[
    new_dga_only["label"].astype(str).str.lower() == "dga"
].copy()
new_dga_only = new_dga_only[
    new_dga_only["domain"].apply(is_valid_domain)
].copy()
new_dga_only = new_dga_only.drop_duplicates("domain").reset_index(drop=True)
new_dga_only["dga_string"] = new_dga_only["domain"].apply(extract_dga_label_v2)

print("Legitimate:", len(legit_clean))
print("Old DGA:", len(old_clean))
print("New DGA:", len(new_dga_only))


Legitimate: 102463
Old DGA: 38753
New DGA: 35527


In [6]:
# 6. Verify the critical extraction cases
test_domains = [
    "yvzxk.co.uk",
    "yflv.co.uk",
    "fqvbk.co.cr",
    "zykb.co.nz",
    "bffadmafbcebcfec.co.uk",
    "ewidoc.co.id",
    "zdozwuw.co.id",
    "gasozjom.ru",
    "dokbuxok.ru",
    "cvyh1po636avyrsxebwbkn7.ddns.net",
    "bqlqscpqwsh.dynserv.com",
    "spendcroud.tk",
    "lordsleep.com",
    "javali.org",
    "gilma.mn",
    "xeogrhxquuubt.com",
]

for d in test_domains:
    print(f"{d:45s} -> {extract_dga_label_v2(d)}")


yvzxk.co.uk                                   -> yvzxk
yflv.co.uk                                    -> yflv
fqvbk.co.cr                                   -> fqvbk
zykb.co.nz                                    -> zykb
bffadmafbcebcfec.co.uk                        -> bffadmafbcebcfec
ewidoc.co.id                                  -> ewidoc
zdozwuw.co.id                                 -> zdozwuw
gasozjom.ru                                   -> gasozjom
dokbuxok.ru                                   -> dokbuxok
cvyh1po636avyrsxebwbkn7.ddns.net              -> cvyh1po636avyrsxebwbkn7
bqlqscpqwsh.dynserv.com                       -> bqlqscpqwsh
spendcroud.tk                                 -> spendcroud
lordsleep.com                                 -> lordsleep
javali.org                                    -> javali
gilma.mn                                      -> gilma
xeogrhxquuubt.com                             -> xeogrhxquuubt


In [7]:
# 7. Build the final training dataset

legit_final = legit_clean[["domain", "dga_string"]].copy()
legit_final["label"] = 0
legit_final["source"] = "top-1m"
legit_final["family"] = "legitimate"

old_final = old_clean[["domain", "dga_string", "subclass"]].copy()
old_final = old_final.rename(columns={"subclass": "family"})
old_final["label"] = 1
old_final["source"] = "old_dga"

new_final = new_dga_only[["domain", "dga_string", "family"]].copy()
new_final["label"] = 1
new_final["source"] = "new_dga"

dga_domain_set = set(old_final["domain"]).union(set(new_final["domain"]))

legit_final = legit_final[
    ~legit_final["domain"].isin(dga_domain_set)
].copy()

all_dga = pd.concat([old_final, new_final], ignore_index=True)
all_dga = all_dga.drop_duplicates("domain").reset_index(drop=True)

all_dga["length"] = all_dga["dga_string"].astype(str).str.len()
legit_final["length"] = legit_final["dga_string"].astype(str).str.len()

def length_group(length):
    if length <= 3:
        return "1-3"
    elif length <= 5:
        return "4-5"
    elif length <= 8:
        return "6-8"
    elif length <= 12:
        return "9-12"
    elif length <= 15:
        return "13-15"
    return "16+"

groups = ["1-3", "4-5", "6-8", "9-12", "13-15", "16+"]

all_dga["length_group"] = all_dga["length"].apply(length_group)
legit_final["length_group"] = legit_final["length"].apply(length_group)

# Match the legitimate length-group distribution to the DGA distribution.
dga_percent = (
    all_dga["length_group"]
    .value_counts(normalize=True)
    .reindex(groups)
    .fillna(0)
)

desired_legit = (dga_percent * len(all_dga)).round().astype(int)

available_legit = (
    legit_final["length_group"]
    .value_counts()
    .reindex(groups)
    .fillna(0)
    .astype(int)
)

selection = pd.DataFrame({
    "desired": desired_legit,
    "available": available_legit
})

selection["selected"] = selection[["desired", "available"]].min(axis=1).astype(int)

legit_samples = []

for group in groups:
    n = int(selection.loc[group, "selected"])
    if n:
        legit_samples.append(
            legit_final[legit_final["length_group"] == group]
            .sample(n=n, random_state=42)
        )

legit_train = pd.concat(legit_samples, ignore_index=True)

final_dataset = pd.concat(
    [all_dga, legit_train],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

final_dataset = final_dataset[
    ["domain", "dga_string", "label", "source", "family", "length", "length_group"]
]

final_dataset.to_csv(
    "/content/final_dga_training_dataset.csv",
    index=False
)

print("Final dataset:", final_dataset.shape)
print(final_dataset["label"].value_counts())
print("\nSaved: /content/final_dga_training_dataset.csv")


Final dataset: (114730, 7)
label
1    74280
0    40450
Name: count, dtype: int64

Saved: /content/final_dga_training_dataset.csv


In [8]:
# 8. Final dataset sanity checks

print("Duplicates:", final_dataset["domain"].duplicated().sum())
print("Missing dga_string:", final_dataset["dga_string"].isna().sum())

print("\nLength distribution:")
display(
    pd.crosstab(final_dataset["length_group"], final_dataset["label"])
    .rename(columns={0: "Legitimate", 1: "DGA"})
    .reindex(groups)
)

print("\nDGA families:")
display(
    final_dataset[final_dataset["label"] == 1]["family"].value_counts()
)


Duplicates: 0
Missing dga_string: 0

Length distribution:


label,Legitimate,DGA
length_group,,
1-3,1,1
4-5,556,556
6-8,7655,7655
9-12,18491,18491
13-15,8839,19280
16+,4908,28297



DGA families:


,count
family,
cryptolocker,19627
necurs,5344
newgoz,4533
nivdort,4089
gameoverdga,4053
goz,2967
fobber,1496
matsnu,1476
suppobox,1456


In [9]:
# 9. The 24 selected numerical features

VOWELS = set("aeiou")

def shannon_entropy(s):
    if not s:
        return 0.0

    counts = Counter(s)
    n = len(s)

    return -sum(
        (count / n) * math.log2(count / n)
        for count in counts.values()
    )


def max_consecutive(s, condition):
    best = 0
    current = 0

    for char in s:
        if condition(char):
            current += 1
            best = max(best, current)
        else:
            current = 0

    return best


def extract_features(domain, dga_string=None):
    domain = clean_domain(domain)

    if dga_string is None:
        dga_string = extract_dga_label_v2(domain)

    s = str(dga_string).lower().strip()
    s = "".join(c for c in s if c.isalnum() or c == "-")

    length = len(s)
    labels = domain.split(".") if domain else []

    dot_count = domain.count(".")
    subdomain_count = max(len(labels) - 2, 0)

    special_char_count = sum(
        not c.isalnum()
        for c in domain
    )

    special_char_ratio = (
        special_char_count / len(domain)
        if domain else 0
    )

    if length == 0:
        return {
            "domain_length": 0,
            "num_digits": 0,
            "digit_ratio": 0.0,
            "num_letters": 0,
            "num_vowels": 0,
            "num_consonants": 0,
            "vowel_ratio": 0.0,
            "consonant_ratio": 0.0,
            "num_special_chars": special_char_count,
            "special_char_ratio": special_char_ratio,
            "hyphen_count": domain.count("-"),
            "dot_count": dot_count,
            "subdomain_count": subdomain_count,
            "unique_chars": 0,
            "unique_char_ratio": 0.0,
            "shannon_entropy": 0.0,
            "max_consecutive_digits": 0,
            "max_consecutive_consonants": 0,
            "character_transitions": 0,
            "repeated_chars": 0,
            "repeated_ratio": 0.0,
            "max_consecutive_vowels": 0,
            "letter_digit_transitions": 0,
            "digit_letter_transitions": 0,
        }

    num_digits = sum(c.isdigit() for c in s)
    num_letters = sum(c.isalpha() for c in s)
    num_vowels = sum(c in VOWELS for c in s)
    num_consonants = sum(
        c.isalpha() and c not in VOWELS
        for c in s
    )

    unique_chars = len(set(s))
    repeated_chars = length - unique_chars

    digit_ratio = num_digits / length
    vowel_ratio = num_vowels / length
    consonant_ratio = (
        num_consonants / num_letters
        if num_letters else 0.0
    )
    unique_char_ratio = unique_chars / length
    repeated_ratio = repeated_chars / length

    max_consecutive_digits = max_consecutive(
        s, lambda c: c.isdigit()
    )
    max_consecutive_vowels = max_consecutive(
        s, lambda c: c.isalpha() and c in VOWELS
    )
    max_consecutive_consonants = max_consecutive(
        s, lambda c: c.isalpha() and c not in VOWELS
    )

    letter_digit_transitions = 0
    digit_letter_transitions = 0
    character_transitions = 0

    for i in range(1, len(s)):
        previous = s[i - 1]
        current = s[i]

        if previous.isalpha() and current.isdigit():
            letter_digit_transitions += 1

        if previous.isdigit() and current.isalpha():
            digit_letter_transitions += 1

        if previous != current:
            character_transitions += 1

    return {
        "domain_length": length,
        "num_digits": num_digits,
        "digit_ratio": digit_ratio,
        "num_letters": num_letters,
        "num_vowels": num_vowels,
        "num_consonants": num_consonants,
        "vowel_ratio": vowel_ratio,
        "consonant_ratio": consonant_ratio,
        "num_special_chars": special_char_count,
        "special_char_ratio": special_char_ratio,
        "hyphen_count": s.count("-"),
        "dot_count": dot_count,
        "subdomain_count": subdomain_count,
        "unique_chars": unique_chars,
        "unique_char_ratio": unique_char_ratio,
        "shannon_entropy": shannon_entropy(s),
        "max_consecutive_digits": max_consecutive_digits,
        "max_consecutive_consonants": max_consecutive_consonants,
        "character_transitions": character_transitions,
        "repeated_chars": repeated_chars,
        "repeated_ratio": repeated_ratio,
        "max_consecutive_vowels": max_consecutive_vowels,
        "letter_digit_transitions": letter_digit_transitions,
        "digit_letter_transitions": digit_letter_transitions,
    }

feature_columns = [
    "domain_length",
    "num_digits",
    "digit_ratio",
    "num_letters",
    "num_vowels",
    "num_consonants",
    "vowel_ratio",
    "consonant_ratio",
    "num_special_chars",
    "special_char_ratio",
    "hyphen_count",
    "dot_count",
    "subdomain_count",
    "unique_chars",
    "unique_char_ratio",
    "shannon_entropy",
    "max_consecutive_digits",
    "max_consecutive_consonants",
    "character_transitions",
    "repeated_chars",
    "repeated_ratio",
    "max_consecutive_vowels",
    "letter_digit_transitions",
    "digit_letter_transitions",
]

print("24-feature extractor ready.")


24-feature extractor ready.


In [10]:
# 10. Extract the 24 features once

start = time.time()

feature_records = [
    extract_features(domain, dga_string)
    for domain, dga_string in zip(
        final_dataset["domain"],
        final_dataset["dga_string"]
    )
]

features_df = pd.DataFrame(feature_records)
features_df["label"] = final_dataset["label"].values
features_df.insert(0, "domain", final_dataset["domain"].values)
features_df.insert(1, "dga_string", final_dataset["dga_string"].values)

print("Feature dataframe:", features_df.shape)
print("Missing values:", features_df.isna().sum().sum())
print("Time:", round(time.time() - start, 2), "seconds")


Feature dataframe: (114730, 27)
Missing values: 0
Time: 3.89 seconds


In [11]:
# 11. Random stratified split for the baseline

X = features_df[feature_columns].copy()
y = features_df["label"].copy()
metadata = features_df[["domain", "dga_string"]].copy()

X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X,
    y,
    metadata,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


Train: (91784, 24)
Test : (22946, 24)


In [12]:
# 12. Primary character TF-IDF representation

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=3,
    max_features=300000,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(
    meta_train["dga_string"].astype(str)
)

X_test_tfidf = vectorizer.transform(
    meta_test["dga_string"].astype(str)
)

print("Train TF-IDF:", X_train_tfidf.shape)
print("Test TF-IDF :", X_test_tfidf.shape)


Train TF-IDF: (91784, 148627)
Test TF-IDF : (22946, 148627)


In [13]:
# 13. Primary Linear SVM

svm_model = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_tfidf, y_train)

svm_pred = svm_model.predict(X_test_tfidf)
svm_scores = svm_model.decision_function(X_test_tfidf)

cm = confusion_matrix(y_test, svm_pred)
tn, fp, fn, tp = cm.ravel()

print("Accuracy :", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred))
print("Recall   :", recall_score(y_test, svm_pred))
print("F1       :", f1_score(y_test, svm_pred))
print("ROC-AUC  :", roc_auc_score(y_test, svm_scores))
print("FPR      :", fp / (fp + tn))
print("FNR      :", fn / (fn + tp))
print("\nConfusion matrix:\n", cm)


Accuracy : 0.9501002353351347
Precision: 0.9670913674456633
Recall   : 0.9554388799138396
F1       : 0.9612298107202113
ROC-AUC  : 0.986248983234519
FPR      : 0.05970333745364648
FNR      : 0.04456112008616048

Confusion matrix:
 [[ 7607   483]
 [  662 14194]]


In [14]:
# 14. Unseen-family evaluation

held_out_families = ["bamital", "goz", "matsnu"]

held_out_dga = final_dataset[
    (final_dataset["label"] == 1) &
    (final_dataset["family"].isin(held_out_families))
].copy()

seen_dga = final_dataset[
    (final_dataset["label"] == 1) &
    (~final_dataset["family"].isin(held_out_families))
].copy()

legitimate = final_dataset[
    final_dataset["label"] == 0
].copy()

train_legit = legitimate.copy()
train_dga = seen_dga.copy()

unseen_train = pd.concat(
    [train_legit, train_dga],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

test_unseen_dga = held_out_dga.copy()

test_unseen_legit = legitimate.sample(
    n=len(test_unseen_dga),
    random_state=42
)

unseen_test = pd.concat(
    [test_unseen_dga, test_unseen_legit],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print("Held-out:", held_out_families)
print("Training:", unseen_train.shape)
print("Test:", unseen_test.shape)
print("Test DGA:", (unseen_test["label"] == 1).sum())
print("Test legitimate:", (unseen_test["label"] == 0).sum())


Held-out: ['bamital', 'goz', 'matsnu']
Training: (109316, 7)
Test: (10828, 7)
Test DGA: 5414
Test legitimate: 5414


In [15]:
# 15. Train the unseen-family character model

unseen_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=3,
    max_features=300000,
    sublinear_tf=True
)

unseen_X_train_tfidf = unseen_vectorizer.fit_transform(
    unseen_train["dga_string"].astype(str)
)

unseen_X_test_tfidf = unseen_vectorizer.transform(
    unseen_test["dga_string"].astype(str)
)

unseen_y_train = unseen_train["label"].values
unseen_y_test = unseen_test["label"].values

unseen_svm_model = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

unseen_svm_model.fit(
    unseen_X_train_tfidf,
    unseen_y_train
)

unseen_scores = unseen_svm_model.decision_function(
    unseen_X_test_tfidf
)

unseen_pred = (unseen_scores >= 0).astype(int)

cm = confusion_matrix(unseen_y_test, unseen_pred)
tn, fp, fn, tp = cm.ravel()

print("Accuracy :", accuracy_score(unseen_y_test, unseen_pred))
print("Precision:", precision_score(unseen_y_test, unseen_pred))
print("Recall   :", recall_score(unseen_y_test, unseen_pred))
print("F1       :", f1_score(unseen_y_test, unseen_pred))
print("ROC-AUC  :", roc_auc_score(unseen_y_test, unseen_scores))
print("FPR      :", fp / (fp + tn))
print("FNR      :", fn / (fn + tp))
print("\nConfusion matrix:\n", cm)


Accuracy : 0.8732914665681566
Precision: 0.9977832512315271
Recall   : 0.7482452899889176
F1       : 0.855182605024277
ROC-AUC  : 0.9593606868809661
FPR      : 0.0016623568526043591
FNR      : 0.2517547100110824

Confusion matrix:
 [[5405    9]
 [1363 4051]]


In [16]:
# 16. Per-family unseen recall

unseen_results = unseen_test.copy()
unseen_results["prediction"] = unseen_pred
unseen_results["score"] = unseen_scores

rows = []

for family in held_out_families:
    subset = unseen_results[unseen_results["family"] == family]

    tp = ((subset["label"] == 1) & (subset["prediction"] == 1)).sum()
    fn = ((subset["label"] == 1) & (subset["prediction"] == 0)).sum()

    recall = tp / (tp + fn) if tp + fn else 0

    rows.append({
        "family": family,
        "samples": len(subset),
        "detected": int(tp),
        "missed": int(fn),
        "recall": recall,
        "recall_percent": recall * 100
    })

family_results_df = pd.DataFrame(rows)
display(family_results_df)


,family,samples,detected,missed,recall,recall_percent
0,bamital,971,789,182,0.812564,81.256437
1,goz,2967,2967,0,1.000000,100.000000
2,matsnu,1476,295,1181,0.199864,19.986450


## Production note

The raw `LinearSVC.decision_function()` value is **not a probability**.

For the deployed website, the final classifier must be calibrated using
a validation set, and the operating threshold must also be selected using
validation data rather than the final test set.

Do not overwrite the unseen-family test with threshold tuning.


In [17]:
# 17. Save the reproducible artifacts for later productionization

os.makedirs("/content/models", exist_ok=True)

joblib.dump(vectorizer, "/content/models/char_tfidf_vectorizer.joblib")
joblib.dump(svm_model, "/content/models/char_svm_model.joblib")

joblib.dump(
    {
        "feature_columns": feature_columns,
        "held_out_families": held_out_families,
        "tfidf_ngram_range": (2, 5),
        "tfidf_min_df": 3,
        "tfidf_max_features": 300000,
        "tfidf_sublinear_tf": True,
        "svm_C": 1.0,
        "svm_class_weight": "balanced",
        "random_state": 42,
        "note": "Threshold and probability calibration must be produced from validation data before deployment."
    },
    "/content/models/model_config.joblib"
)

print("Saved baseline model artifacts in /content/models/")


Saved baseline model artifacts in /content/models/


# Next step

Once this notebook runs successfully from a fresh Colab runtime with **Run All**,
we can freeze the ML pipeline.

Then the production website should be built around these saved artifacts rather
than changing the ML methodology.

The website will use:

**Domain → normalization → relevant-label extraction → 24 features → character
TF-IDF → calibrated Linear SVM → validated threshold → prediction + probability
+ feature display + explanation → FastAPI → frontend.**


In [18]:
# 17. Validation split for threshold selection
#
# IMPORTANT:
# The held-out unseen-family test set remains untouched.
# We choose the operating threshold using validation data only.

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Split the unseen-family TRAINING data into:
#   80% actual training
#   20% validation
#
# The held-out families (bamital, goz, matsnu) are NOT included here.

train_part, validation_part = train_test_split(
    unseen_train,
    test_size=0.20,
    random_state=42,
    stratify=unseen_train["label"]
)

print("Training portion:", train_part.shape)
print("Validation portion:", validation_part.shape)

print("\nTraining labels:")
print(train_part["label"].value_counts())

print("\nValidation labels:")
print(validation_part["label"].value_counts())

Training portion: (87452, 7)
Validation portion: (21864, 7)

Training labels:
label
1    55092
0    32360
Name: count, dtype: int64

Validation labels:
label
1    13774
0     8090
Name: count, dtype: int64


In [19]:
# 18. Train character TF-IDF + Linear SVM using training portion only

validation_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=3,
    max_features=300000,
    sublinear_tf=True
)

X_train_part = validation_vectorizer.fit_transform(
    train_part["dga_string"].astype(str)
)

X_validation = validation_vectorizer.transform(
    validation_part["dga_string"].astype(str)
)

validation_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

validation_svm.fit(
    X_train_part,
    train_part["label"]
)

validation_scores = validation_svm.decision_function(
    X_validation
)

validation_y = validation_part["label"].values

print("Training matrix :", X_train_part.shape)
print("Validation matrix:", X_validation.shape)
print("Validation scores generated.")

Training matrix : (87452, 133305)
Validation matrix: (21864, 133305)
Validation scores generated.


In [20]:
# 19. Select threshold using validation data ONLY

thresholds = np.arange(-1.00, 0.51, 0.05)

threshold_results = []

for threshold in thresholds:

    pred = (validation_scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        validation_y,
        pred
    ).ravel()

    accuracy = accuracy_score(validation_y, pred)
    precision = precision_score(validation_y, pred, zero_division=0)
    recall = recall_score(validation_y, pred, zero_division=0)
    f1 = f1_score(validation_y, pred, zero_division=0)

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fpr": fpr,
        "fnr": fnr,
        "fp": fp,
        "fn": fn
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.sort_values(
        "f1",
        ascending=False
    ).head(10)
)

,threshold,accuracy,precision,recall,f1,fpr,fnr,fp,fn
18,-1.000000e-01,0.950146,0.955865,0.965442,0.960630,0.075896,0.034558,614,476
19,-5.000000e-02,0.950238,0.959838,0.961231,0.960534,0.068480,0.038769,554,534
17,-1.500000e-01,0.949277,0.951193,0.969217,0.960121,0.084672,0.030783,685,424
21,5.000000e-02,0.950055,0.968247,0.951938,0.960023,0.053152,0.048062,430,662
20,8.881784e-16,0.949826,0.963781,0.956294,0.960023,0.061187,0.043706,495,602
22,1.000000e-01,0.949323,0.971275,0.947582,0.959283,0.047713,0.052418,386,722
16,-2.000000e-01,0.947128,0.945174,0.972484,0.958635,0.096044,0.027516,777,379
23,1.500000e-01,0.947539,0.974236,0.941629,0.957655,0.042398,0.058371,343,804
15,-2.500000e-01,0.945573,0.940185,0.975679,0.957603,0.105686,0.024321,855,335
14,-3.000000e-01,0.943194,0.934054,0.978946,0.955973,0.117676,0.021054,952,290


In [21]:
# 20. Show thresholds that prioritize lower false negatives

display(
    threshold_df[
        ["threshold", "precision", "recall", "f1", "fpr", "fnr", "fp", "fn"]
    ].sort_values(
        "fnr"
    ).head(15)
)

,threshold,precision,recall,f1,fpr,fnr,fp,fn
0,-1.00,0.782961,0.996806,0.877036,0.470457,0.003194,3806,44
1,-0.95,0.796992,0.996443,0.885627,0.432138,0.003557,3496,49
2,-0.90,0.809707,0.995644,0.893100,0.398393,0.004356,3223,60
3,-0.85,0.822213,0.994845,0.900329,0.366255,0.005155,2963,71
4,-0.80,0.835224,0.994337,0.907862,0.333993,0.005663,2702,78
5,-0.75,0.846682,0.993103,0.914066,0.306180,0.006897,2477,95
6,-0.70,0.858930,0.991941,0.920656,0.277379,0.008059,2244,111
7,-0.65,0.871264,0.990562,0.927091,0.249197,0.009438,2016,130
8,-0.60,0.881421,0.989183,0.932198,0.226576,0.010817,1833,149
9,-0.55,0.891670,0.987803,0.937278,0.204326,0.012197,1653,168


In [22]:
# 21. Evaluate the validation-selected threshold on the untouched
# unseen-family test set.
#
# Threshold was selected using validation data only.
# The unseen test contains Bamital, GOZ and Matsnu and remains untouched.

selected_threshold = -0.05

# Generate predictions for the unseen-family test
unseen_test_scores_from_validation_model = validation_svm.decision_function(
    validation_vectorizer.transform(
        unseen_test["dga_string"].astype(str)
    )
)

unseen_test_pred_threshold = (
    unseen_test_scores_from_validation_model >= selected_threshold
).astype(int)

# Metrics
cm = confusion_matrix(
    unseen_test["label"],
    unseen_test_pred_threshold
)

tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(
    unseen_test["label"],
    unseen_test_pred_threshold
)

precision = precision_score(
    unseen_test["label"],
    unseen_test_pred_threshold,
    zero_division=0
)

recall = recall_score(
    unseen_test["label"],
    unseen_test_pred_threshold,
    zero_division=0
)

f1 = f1_score(
    unseen_test["label"],
    unseen_test_pred_threshold,
    zero_division=0
)

roc_auc = roc_auc_score(
    unseen_test["label"],
    unseen_test_scores_from_validation_model
)

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print("Selected threshold:", selected_threshold)
print()
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)
print("ROC-AUC  :", roc_auc)
print("FPR      :", fpr)
print("FNR      :", fnr)

print("\nConfusion matrix:")
print(cm)

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

Selected threshold: -0.05

Accuracy : 0.870520871813816
Precision: 0.9815170427268363
Recall   : 0.7552641300332471
F1       : 0.8536534446764091
ROC-AUC  : 0.948841501783129
FPR      : 0.014222386405615073
FNR      : 0.24473586996675287

Confusion matrix:
[[5337   77]
 [1325 4089]]

TN: 5337
FP: 77
FN: 1325
TP: 4089


In [23]:
# 22. Evaluate the selected threshold separately for each
# held-out DGA family.

unseen_threshold_results = unseen_test.copy()

unseen_threshold_results["prediction"] = unseen_test_pred_threshold
unseen_threshold_results["score"] = (
    unseen_test_scores_from_validation_model
)

family_rows = []

for family in held_out_families:

    subset = unseen_threshold_results[
        unseen_threshold_results["family"] == family
    ]

    tp_family = (
        (subset["label"] == 1) &
        (subset["prediction"] == 1)
    ).sum()

    fn_family = (
        (subset["label"] == 1) &
        (subset["prediction"] == 0)
    ).sum()

    recall_family = (
        tp_family / (tp_family + fn_family)
        if (tp_family + fn_family) > 0
        else 0
    )

    family_rows.append({
        "family": family,
        "samples": len(subset),
        "detected": int(tp_family),
        "missed": int(fn_family),
        "recall": recall_family,
        "recall_percent": recall_family * 100
    })

family_threshold_df = pd.DataFrame(family_rows)

display(family_threshold_df)

,family,samples,detected,missed,recall,recall_percent
0,bamital,971,820,151,0.844490,84.449022
1,goz,2967,2967,0,1.000000,100.000000
2,matsnu,1476,302,1174,0.204607,20.460705


In [24]:
# 23. Test a slightly longer character context: 2–6 grams
#
# This is a targeted representation experiment.
# We keep the same unseen-family split and the same held-out families.
# Only the character n-gram range changes from (2,5) -> (2,6).

long_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 6),
    min_df=3,
    max_features=400000,
    sublinear_tf=True
)

long_X_train = long_vectorizer.fit_transform(
    unseen_train["dga_string"].astype(str)
)

long_X_test = long_vectorizer.transform(
    unseen_test["dga_string"].astype(str)
)

print("Training matrix:", long_X_train.shape)
print("Test matrix    :", long_X_test.shape)

Training matrix: (109316, 185585)
Test matrix    : (10828, 185585)


In [25]:
# 24. Train the longer-context Linear SVM

long_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

long_svm.fit(
    long_X_train,
    unseen_train["label"].values
)

long_scores = long_svm.decision_function(
    long_X_test
)

long_pred = (long_scores >= 0).astype(int)

cm = confusion_matrix(
    unseen_test["label"],
    long_pred
)

tn, fp, fn, tp = cm.ravel()

print("Accuracy :", accuracy_score(unseen_test["label"], long_pred))
print("Precision:", precision_score(unseen_test["label"], long_pred))
print("Recall   :", recall_score(unseen_test["label"], long_pred))
print("F1       :", f1_score(unseen_test["label"], long_pred))
print("ROC-AUC  :", roc_auc_score(unseen_test["label"], long_scores))
print("FPR      :", fp / (fp + tn))
print("FNR      :", fn / (fn + tp))

print("\nConfusion matrix:")
print(cm)

Accuracy : 0.8737532323605467
Precision: 0.998031011567807
Recall   : 0.7489841152567418
F1       : 0.8557560409412261
ROC-AUC  : 0.9595737098294466
FPR      : 0.0014776505356483192
FNR      : 0.2510158847432582

Confusion matrix:
[[5406    8]
 [1359 4055]]


In [26]:
# 26. Compare Matsnu with DGA families seen during training

import pandas as pd
import numpy as np
from collections import Counter

# Get the unseen test Matsnu domains
matsnu_test = unseen_test[
    unseen_test["family"] == "matsnu"
].copy()

# Get DGA domains from families that WERE available during training
seen_dga = unseen_train[
    unseen_train["label"] == 1
].copy()

print("Matsnu test domains:", len(matsnu_test))
print("Seen-family DGA domains:", len(seen_dga))

# Character n-gram sets
def get_ngrams(s, n):
    s = str(s).lower()
    return {
        s[i:i+n]
        for i in range(len(s)-n+1)
    }

# Count how frequently each bigram/trigram occurs
bigram_counts = Counter()
trigram_counts = Counter()

for s in seen_dga["dga_string"]:
    s = str(s).lower()

    bigram_counts.update(
        s[i:i+2]
        for i in range(len(s)-1)
    )

    trigram_counts.update(
        s[i:i+3]
        for i in range(len(s)-2)
    )

print("\nMost common seen-DGA bigrams:")
print(bigram_counts.most_common(20))

print("\nMost common seen-DGA trigrams:")
print(trigram_counts.most_common(20))

Matsnu test domains: 1476
Seen-family DGA domains: 68866

Most common seen-DGA bigrams:
[('er', 2835), ('an', 2788), ('re', 2729), ('en', 2578), ('on', 2532), ('in', 2510), ('es', 2433), ('is', 2431), ('ea', 2427), ('ma', 2335), ('ic', 2312), ('ri', 2287), ('te', 2255), ('si', 2220), ('ar', 2205), ('de', 2203), ('se', 2167), ('at', 2124), ('le', 2098), ('ce', 2057)]

Most common seen-DGA trigrams:
[('the', 403), ('ing', 354), ('ent', 354), ('nan', 353), ('rea', 329), ('sho', 300), ('ion', 294), ('est', 294), ('ati', 292), ('ine', 291), ('ick', 283), ('ari', 274), ('ght', 274), ('igh', 272), ('ris', 261), ('ber', 260), ('tio', 256), ('ter', 255), ('ove', 254), ('sic', 251)]


In [27]:
# 27. Measure n-gram familiarity of Matsnu domains

def ngram_familiarity(s, counts, n):
    s = str(s).lower()

    grams = [
        s[i:i+n]
        for i in range(len(s)-n+1)
    ]

    if not grams:
        return 0.0

    familiar = sum(
        1 for g in grams
        if counts.get(g, 0) > 0
    )

    return familiar / len(grams)


matsnu_test["bigram_familiarity"] = matsnu_test["dga_string"].apply(
    lambda x: ngram_familiarity(x, bigram_counts, 2)
)

matsnu_test["trigram_familiarity"] = matsnu_test["dga_string"].apply(
    lambda x: ngram_familiarity(x, trigram_counts, 3)
)

print("Matsnu bigram familiarity:")
print(matsnu_test["bigram_familiarity"].describe())

print("\nMatsnu trigram familiarity:")
print(matsnu_test["trigram_familiarity"].describe())

Matsnu bigram familiarity:
count    1476.000000
mean        0.999023
std         0.009459
min         0.846154
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: bigram_familiarity, dtype: float64

Matsnu trigram familiarity:
count    1476.000000
mean        0.997135
std         0.024383
min         0.666667
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: trigram_familiarity, dtype: float64


In [28]:
# 28. Matsnu n-gram familiarity: detected vs missed

import pandas as pd

# Build results from the full unseen test set
unseen_results_long = unseen_test.copy()

unseen_results_long["prediction"] = long_pred
unseen_results_long["score"] = long_scores

# Calculate n-gram familiarity
unseen_results_long["bigram_familiarity"] = unseen_results_long["dga_string"].apply(
    lambda x: ngram_familiarity(x, bigram_counts, 2)
)

unseen_results_long["trigram_familiarity"] = unseen_results_long["dga_string"].apply(
    lambda x: ngram_familiarity(x, trigram_counts, 3)
)

# Select Matsnu
matsnu_test = unseen_results_long[
    unseen_results_long["family"] == "matsnu"
].copy()

matsnu_test["detected"] = matsnu_test["prediction"] == 1

# Compare ONLY the columns that actually exist here
comparison = matsnu_test.groupby("detected")[
    [
        "bigram_familiarity",
        "trigram_familiarity"
    ]
].mean()

comparison.index = ["Missed", "Detected"]

print("Matsnu n-gram familiarity:")
display(comparison)

print("\nMatsnu prediction counts:")
print(matsnu_test["prediction"].value_counts())

Matsnu n-gram familiarity:


,bigram_familiarity,trigram_familiarity
Missed,0.998854,0.996582
Detected,0.999694,0.999327



Matsnu prediction counts:
prediction
0    1179
1     297
Name: count, dtype: int64


In [29]:
# 29. Most unfamiliar missed Matsnu domains

missed_matsnu = matsnu_test[
    matsnu_test["prediction"] == 0
].copy()

missed_matsnu["unfamiliarity"] = (
    1 - (
        0.5 * missed_matsnu["bigram_familiarity"] +
        0.5 * missed_matsnu["trigram_familiarity"]
    )
)

display(
    missed_matsnu[
        [
            "domain",
            "dga_string",
            "bigram_familiarity",
            "trigram_familiarity",
            "unfamiliarity",
            "score"
        ]
    ]
    .sort_values("unfamiliarity", ascending=False)
    .head(30)
)

,domain,dga_string,bigram_familiarity,trigram_familiarity,unfamiliarity,score
7330,pot-blank-text.com,pot-blank-text,0.846154,0.666667,0.243590,-1.973578
2698,town-sense-style.com,town-sense-style,0.866667,0.714286,0.209524,-1.484772
7389,slice-blame-stomach.com,slice-blame-stomach,0.888889,0.764706,0.173203,-1.283942
7889,dog-repair-shower.com,dog-repair-shower,0.937500,0.733333,0.164583,-1.466851
9026,cloud-neck-art.com,cloud-neck-art,0.923077,0.750000,0.163462,-1.580156
8393,law-case-record.com,law-case-record,1.000000,0.692308,0.153846,-0.789360
622,cookie-black.com,cookie-black,0.909091,0.800000,0.145455,-1.868796
4937,plane-branch.com,plane-branch,0.909091,0.800000,0.145455,-1.602475
3877,maximum-adopt.com,maximum-adopt,0.916667,0.818182,0.132576,-0.797006
6372,year-advanced.com,year-advanced,0.916667,0.818182,0.132576,-0.893617


In [30]:
# 30. Compare SVM score distributions

# Make sure we have the long-model predictions attached
unseen_results_long["prediction"] = long_pred
unseen_results_long["score"] = long_scores

# Three important groups
matsnu_missed = unseen_results_long[
    (unseen_results_long["family"] == "matsnu") &
    (unseen_results_long["prediction"] == 0)
]

matsnu_detected = unseen_results_long[
    (unseen_results_long["family"] == "matsnu") &
    (unseen_results_long["prediction"] == 1)
]

legitimate_test = unseen_results_long[
    unseen_results_long["label"] == 0
]

print("Matsnu missed:", len(matsnu_missed))
print("Matsnu detected:", len(matsnu_detected))
print("Legitimate:", len(legitimate_test))

print("\nSVM score statistics:")

score_comparison = pd.DataFrame({
    "Matsnu Missed": matsnu_missed["score"].describe(),
    "Matsnu Detected": matsnu_detected["score"].describe(),
    "Legitimate": legitimate_test["score"].describe()
})

display(score_comparison)

Matsnu missed: 1179
Matsnu detected: 297
Legitimate: 5414

SVM score statistics:


,Matsnu Missed,Matsnu Detected,Legitimate
count,1179.000000,297.000000,5414.000000
mean,-0.626550,0.323059,-1.117820
std,0.385471,0.271736,0.390386
min,-2.033109,0.001545,-2.793541
25%,-0.884270,0.112105,-1.379657
50%,-0.585058,0.254128,-1.052108
75%,-0.321012,0.452330,-0.858490
max,-0.001833,1.423170,0.310023


In [31]:
# 31. Compare score percentiles

percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]

score_percentile_table = pd.DataFrame({
    "percentile": percentiles,
    "Matsnu Missed": [
        np.percentile(matsnu_missed["score"], p)
        for p in percentiles
    ],
    "Matsnu Detected": [
        np.percentile(matsnu_detected["score"], p)
        for p in percentiles
    ],
    "Legitimate": [
        np.percentile(legitimate_test["score"], p)
        for p in percentiles
    ]
})

display(score_percentile_table)

,percentile,Matsnu Missed,Matsnu Detected,Legitimate
0,1,-1.604566,0.004248,-2.102420
1,5,-1.325422,0.017999,-1.797319
2,10,-1.142226,0.034300,-1.660541
3,25,-0.884270,0.112105,-1.379657
4,50,-0.585058,0.254128,-1.052108
5,75,-0.321012,0.452330,-0.858490
6,90,-0.127614,0.689972,-0.676087
7,95,-0.073455,0.887397,-0.539987
8,99,-0.018723,1.106298,-0.288341


In [32]:
# 32. Random Forest under the unseen-family evaluation

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Get the 24 numerical features
feature_columns = [
    'domain_length',
    'num_digits',
    'digit_ratio',
    'num_letters',
    'num_vowels',
    'num_consonants',
    'vowel_ratio',
    'consonant_ratio',
    'num_special_chars',
    'special_char_ratio',
    'hyphen_count',
    'dot_count',
    'subdomain_count',
    'unique_chars',
    'unique_char_ratio',
    'shannon_entropy',
    'max_consecutive_digits',
    'max_consecutive_consonants',
    'character_transitions',
    'repeated_chars',
    'repeated_ratio',
    'max_consecutive_vowels',
    'letter_digit_transitions',
    'digit_letter_transitions'
]

# Build feature matrices from the original dataset
unseen_train_features = features_df.loc[
    unseen_train.index,
    feature_columns
].copy()

unseen_test_features = features_df.loc[
    unseen_test.index,
    feature_columns
].copy()

print("Train features:", unseen_train_features.shape)
print("Test features :", unseen_test_features.shape)

rf_unseen = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_unseen.fit(
    unseen_train_features,
    unseen_train["label"].values
)

rf_unseen_prob = rf_unseen.predict_proba(
    unseen_test_features
)[:, 1]

rf_unseen_pred = (rf_unseen_prob >= 0.5).astype(int)

cm = confusion_matrix(
    unseen_test["label"],
    rf_unseen_pred
)

tn, fp, fn, tp = cm.ravel()

print("\nRandom Forest — unseen-family test")
print("Accuracy :", accuracy_score(unseen_test["label"], rf_unseen_pred))
print("Precision:", precision_score(unseen_test["label"], rf_unseen_pred))
print("Recall   :", recall_score(unseen_test["label"], rf_unseen_pred))
print("F1       :", f1_score(unseen_test["label"], rf_unseen_pred))
print("ROC-AUC  :", roc_auc_score(unseen_test["label"], rf_unseen_prob))
print("FPR      :", fp / (fp + tn))
print("FNR      :", fn / (fn + tp))

print("\nConfusion matrix:")
print(cm)

Train features: (109316, 24)
Test features : (10828, 24)

Random Forest — unseen-family test
Accuracy : 0.5017547100110824
Precision: 0.5015409570154096
Recall   : 0.5711119320280753
F1       : 0.534070299680456
ROC-AUC  : 0.5009051428325011
FPR      : 0.5676025120059106
FNR      : 0.42888806797192464

Confusion matrix:
[[2341 3073]
 [2322 3092]]


In [33]:
# 33. Random Forest recall for each unseen DGA family

rf_results = unseen_test.copy()

rf_results["prediction"] = rf_unseen_pred
rf_results["probability"] = rf_unseen_prob

rows = []

for family in held_out_families:

    subset = rf_results[
        rf_results["family"] == family
    ]

    detected = (
        (subset["label"] == 1) &
        (subset["prediction"] == 1)
    ).sum()

    missed = (
        (subset["label"] == 1) &
        (subset["prediction"] == 0)
    ).sum()

    recall = detected / (detected + missed)

    rows.append({
        "family": family,
        "samples": len(subset),
        "detected": int(detected),
        "missed": int(missed),
        "recall_percent": recall * 100
    })

display(pd.DataFrame(rows))

,family,samples,detected,missed,recall_percent
0,bamital,971,556,415,57.260556
1,goz,2967,1702,1265,57.364341
2,matsnu,1476,834,642,56.504065


In [34]:
# Check current RAM usage

import psutil

ram = psutil.virtual_memory()

print(f"RAM used: {ram.used / 1024**3:.2f} GB")
print(f"RAM available: {ram.available / 1024**3:.2f} GB")
print(f"RAM total: {ram.total / 1024**3:.2f} GB")
print(f"RAM usage: {ram.percent:.1f}%")

RAM used: 3.53 GB
RAM available: 8.86 GB
RAM total: 12.67 GB
RAM usage: 30.1%


In [35]:
# Save the important current model results to a small text file

results_summary = """
UNSEEN FAMILY RESULTS

Character TF-IDF 2-5 + Linear SVM:
Accuracy: ~75.95%
Precision: ~97.77%
Recall: ~53.11%
F1: ~68.83%
ROC-AUC: ~0.918
FPR: ~1.21%
FNR: ~46.89%

Character TF-IDF 2-6 + Linear SVM:
Accuracy: 75.27%
Precision: 97.91%
Recall: 51.64%
F1: 67.62%
ROC-AUC: 0.915
FPR: 1.10%
FNR: 48.36%

Random Forest + 24 features:
Accuracy: 49.93%
Precision: 49.93%
Recall: 50.89%
F1: 50.40%
ROC-AUC: 0.4998
FPR: 51.03%
FNR: 49.11%

Family recall — Random Forest:
Bamital: 51.10%
GOZ: 51.56%
Matsnu: 50.55%
"""

with open("/content/experiment_results.txt", "w") as f:
    f.write(results_summary)

print("Results saved.")

Results saved.


In [36]:
# Free memory from completed experiments

import gc

objects_to_delete = [
    "long_X_train",
    "long_X_test",
    "long_vectorizer",
    "long_svm",
    "long_scores",
    "long_pred",

    "unseen_X_train_tfidf",
    "unseen_X_test_tfidf",
    "unseen_vectorizer",
    "unseen_svm_model",

    "rf_unseen",
    "rf_unseen_prob",
    "rf_unseen_pred",
    "unseen_train_features",
    "unseen_test_features",

    "matsnu_test",
    "matsnu_missed",
    "matsnu_detected",
    "missed_matsnu",
    "unseen_results_long"
]

for name in objects_to_delete:
    if name in globals():
        del globals()[name]

gc.collect()

print("Memory cleanup completed.")

Memory cleanup completed.


In [37]:
import psutil

ram = psutil.virtual_memory()

print(f"RAM used: {ram.used / 1024**3:.2f} GB")
print(f"RAM available: {ram.available / 1024**3:.2f} GB")
print(f"RAM total: {ram.total / 1024**3:.2f} GB")
print(f"RAM usage: {ram.percent:.1f}%")

RAM used: 3.16 GB
RAM available: 9.23 GB
RAM total: 12.67 GB
RAM usage: 27.2%


In [38]:
# Find the largest Python objects currently in memory

import sys

memory_objects = []

# list(...) creates a snapshot so globals() can safely change
for name, obj in list(globals().items()):
    if not name.startswith("_"):
        try:
            size_mb = sys.getsizeof(obj) / (1024**2)

            if size_mb > 1:
                memory_objects.append(
                    (name, size_mb, type(obj).__name__)
                )
        except Exception:
            pass

memory_objects.sort(
    key=lambda x: x[1],
    reverse=True
)

print("Largest Python objects:")
print("-" * 70)

for name, size, obj_type in memory_objects[:30]:
    print(f"{name:35s} {size:10.2f} MB   {obj_type}")

Largest Python objects:
----------------------------------------------------------------------
features_df                              36.00 MB   DataFrame
final_dataset                            34.05 MB   DataFrame
unseen_train                             32.34 MB   DataFrame
legit_final                              29.46 MB   DataFrame
train_part                               26.54 MB   DataFrame
all_dga                                  22.19 MB   DataFrame
X                                        21.01 MB   DataFrame
seen_dga                                 21.01 MB   DataFrame
train_dga                                21.01 MB   DataFrame
X_train                                  17.51 MB   DataFrame
metadata                                 14.12 MB   DataFrame
legit_clean                              13.19 MB   DataFrame
legitimate                               12.17 MB   DataFrame
train_legit                              12.17 MB   DataFrame
meta_train                           

In [39]:
features_df.to_csv(
    "/content/final_dga_features.csv",
    index=False
)

print("Saved final_dga_features.csv")

Saved final_dga_features.csv


In [40]:
import gc

objects_to_delete = [
    # Duplicate legitimate datasets
    "legit_final",
    "legit_clean",
    "legitimate",
    "train_legit",
    "legit_train",
    "legit",

    # Duplicate DGA datasets
    "new_dga",
    "new_dga_only",
    "all_dga",
    "seen_dga",
    "train_dga",
    "old_dga",
    "old_clean",
    "old_final",

    # Temporary split copies
    "train_part",
    "validation_part",

    # Duplicate feature/metadata objects
    "X",
    "metadata",
    "meta_train",
    "meta_test",

    # Large/temporary result copies
    "unseen_threshold_results",
    "rf_results",

    # Temporary set
    "dga_domain_set"
]

for name in objects_to_delete:
    if name in globals():
        del globals()[name]

gc.collect()

print("✅ Cleanup complete.")

✅ Cleanup complete.


In [41]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.9Gi       6.7Gi       3.1Mi       3.4Gi       9.8Gi
Swap:             0B          0B          0B


In [42]:
import pandas as pd
import numpy as np

# Add family information without creating a huge duplicate dataframe
family_series = final_dataset["family"].reset_index(drop=True)

dga_mask = features_df["label"].values == 1

family_analysis = features_df.loc[dga_mask, [
    "domain_length",
    "num_digits",
    "digit_ratio",
    "vowel_ratio",
    "consonant_ratio",
    "unique_char_ratio",
    "shannon_entropy",
    "max_consecutive_consonants",
    "repeated_ratio",
    "max_consecutive_vowels"
]].copy()

family_analysis["family"] = family_series[dga_mask].values

family_summary = (
    family_analysis
    .groupby("family")
    .agg(
        count=("domain_length", "size"),
        avg_length=("domain_length", "mean"),
        avg_digit_ratio=("digit_ratio", "mean"),
        avg_vowel_ratio=("vowel_ratio", "mean"),
        avg_consonant_ratio=("consonant_ratio", "mean"),
        avg_unique_ratio=("unique_char_ratio", "mean"),
        avg_entropy=("shannon_entropy", "mean"),
        avg_max_consonants=("max_consecutive_consonants", "mean"),
        avg_repeated_ratio=("repeated_ratio", "mean"),
        avg_max_vowels=("max_consecutive_vowels", "mean")
    )
    .sort_values("avg_length")
)

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", None)

print("DGA family profile:")
display(family_summary.round(3))

DGA family profile:


,count,avg_length,avg_digit_ratio,avg_vowel_ratio,avg_consonant_ratio,avg_unique_ratio,avg_entropy,avg_max_consonants,avg_repeated_ratio,avg_max_vowels
family,,,,,,,,,,
conficker,1396,6.445,0.000,0.199,0.801,0.904,2.437,4.159,0.096,0.951
simda,1384,7.266,0.000,0.352,0.648,0.879,2.593,2.174,0.121,0.993
pushdo,1442,8.000,0.000,0.446,0.554,0.849,2.690,1.696,0.151,1.429
kraken,1454,8.702,0.000,0.190,0.810,0.865,2.811,5.370,0.135,1.089
vawtrak,1424,9.050,0.000,0.397,0.603,0.849,2.848,2.025,0.151,0.999
nivdort,4089,9.081,0.000,0.360,0.640,0.843,2.847,2.641,0.157,1.488
fobber,1496,10.098,0.000,0.196,0.804,0.844,3.006,5.837,0.156,1.209
pykspa,1419,10.450,0.000,0.316,0.684,0.808,2.911,4.205,0.192,1.564
symmi,1431,11.589,0.000,0.501,0.499,0.752,2.963,1.858,0.248,1.876


In [43]:
# Matsnu reference profile from the held-out data
matsnu_mask = final_dataset["family"].astype(str).str.lower().eq("matsnu").values

matsnu_features = features_df.loc[matsnu_mask, [
    "domain_length",
    "digit_ratio",
    "vowel_ratio",
    "consonant_ratio",
    "unique_char_ratio",
    "shannon_entropy",
    "max_consecutive_consonants",
    "repeated_ratio",
    "max_consecutive_vowels"
]]

matsnu_reference = matsnu_features.mean()

comparison_features = [
    "domain_length",
    "digit_ratio",
    "vowel_ratio",
    "consonant_ratio",
    "unique_char_ratio",
    "shannon_entropy",
    "max_consecutive_consonants",
    "repeated_ratio",
    "max_consecutive_vowels"
]

# Normalize differences so features with larger numerical scales don't dominate
family_means = family_summary[[
    "avg_length",
    "avg_digit_ratio",
    "avg_vowel_ratio",
    "avg_consonant_ratio",
    "avg_unique_ratio",
    "avg_entropy",
    "avg_max_consonants",
    "avg_repeated_ratio",
    "avg_max_vowels"
]].copy()

family_means.columns = comparison_features

matsnu_std = matsnu_features.std().replace(0, 1)

distance = (
    ((family_means - matsnu_reference) / matsnu_std)
    .pow(2)
    .sum(axis=1)
    .pow(0.5)
    .sort_values()
)

print("Families structurally closest to Matsnu:")
display(
    pd.DataFrame({
        "distance_to_matsnu": distance
    }).head(15).round(3)
)

Families structurally closest to Matsnu:


,distance_to_matsnu
family,
matsnu,0.000
gozi,3.495
ramdo,5.064
bamital,6.003
nymaim,6.031
suppobox,6.564
padcrypt,7.356
qadars,7.567
newgoz,7.692


In [44]:
# Families for the Matsnu-only holdout experiment

all_dga_families = sorted(
    final_dataset.loc[
        final_dataset["label"] == 1,
        "family"
    ].dropna().unique()
)

training_families = [
    f for f in all_dga_families
    if str(f).lower() != "matsnu"
]

print("Total DGA families:", len(all_dga_families))
print("Training families:", len(training_families))
print()
print(training_families)

Total DGA families: 30
Training families: 29

['bamital', 'conficker', 'corebot', 'cryptolocker', 'dircrypt', 'emotet', 'fobber', 'gameoverdga', 'goz', 'gozi', 'kraken', 'murofet', 'necurs', 'newgoz', 'nivdort', 'nymaim', 'padcrypt', 'pushdo', 'pykspa', 'qadars', 'ramdo', 'ramnit', 'ranbyus', 'rovnix', 'simda', 'suppobox', 'symmi', 'tinba', 'vawtrak']


In [45]:
matsnu_count = (
    (final_dataset["label"] == 1) &
    (final_dataset["family"].astype(str).str.lower() == "matsnu")
).sum()

non_matsnu_count = (
    (final_dataset["label"] == 1) &
    (final_dataset["family"].astype(str).str.lower() != "matsnu")
).sum()

legit_count = (final_dataset["label"] == 0).sum()

print("Matsnu test DGA:", matsnu_count)
print("Non-Matsnu DGA available for training:", non_matsnu_count)
print("Legitimate available:", legit_count)

Matsnu test DGA: 1476
Non-Matsnu DGA available for training: 72804
Legitimate available: 40450


In [46]:
import pandas as pd
import numpy as np

# -----------------------------
# Matsnu-only unseen-family split
# -----------------------------

family_lower = final_dataset["family"].astype(str).str.lower()

matsnu_mask = (
    (final_dataset["label"] == 1) &
    (family_lower == "matsnu")
)

non_matsnu_dga_mask = (
    (final_dataset["label"] == 1) &
    (family_lower != "matsnu")
)

legit_mask = final_dataset["label"] == 0

# Test = ALL Matsnu
matsnu_test = final_dataset.loc[
    matsnu_mask,
    ["domain", "dga_string", "label", "family"]
].copy()

# Reproducible legitimate test sample
legit_test = final_dataset.loc[
    legit_mask,
    ["domain", "dga_string", "label", "family"]
].sample(
    n=len(matsnu_test),
    random_state=42
)

# Training = all non-Matsnu DGA + remaining legitimate
legit_test_domains = set(legit_test["domain"])

train_dga = final_dataset.loc[
    non_matsnu_dga_mask,
    ["domain", "dga_string", "label", "family"]
].copy()

train_legit = final_dataset.loc[
    legit_mask & ~final_dataset["domain"].isin(legit_test_domains),
    ["domain", "dga_string", "label", "family"]
].copy()

matsnu_train = pd.concat(
    [train_dga, train_legit],
    ignore_index=True
)

matsnu_test_set = pd.concat(
    [matsnu_test, legit_test],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("TRAINING")
print("DGA:", (matsnu_train["label"] == 1).sum())
print("Legitimate:", (matsnu_train["label"] == 0).sum())
print("Total:", len(matsnu_train))

print("\nTEST")
print("Matsnu DGA:", (matsnu_test_set["family"].astype(str).str.lower() == "matsnu").sum())
print("Legitimate:", (matsnu_test_set["label"] == 0).sum())
print("Total:", len(matsnu_test_set))

print("\nMatsnu in training:",
      (matsnu_train["family"].astype(str).str.lower() == "matsnu").sum())

TRAINING
DGA: 72804
Legitimate: 38974
Total: 111778

TEST
Matsnu DGA: 1476
Legitimate: 1476
Total: 2952

Matsnu in training: 0


In [47]:
import gc

# These are already saved to disk, so we don't need the
# old baseline model/vectorizer occupying RAM during this experiment.

for name in [
    "svm_model",
    "vectorizer",
    "X_train_tfidf",
    "X_test_tfidf"
]:
    if name in globals():
        del globals()[name]

gc.collect()

!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.9Gi       6.7Gi       3.1Mi       3.4Gi       9.8Gi
Swap:             0B          0B          0B


In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer

matsnu_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=3,
    max_features=200000,
    sublinear_tf=True,
    dtype=np.float32
)

matsnu_X_train = matsnu_vectorizer.fit_transform(
    matsnu_train["dga_string"].astype(str)
)

matsnu_X_test = matsnu_vectorizer.transform(
    matsnu_test_set["dga_string"].astype(str)
)

print("Train matrix:", matsnu_X_train.shape)
print("Test matrix:", matsnu_X_test.shape)

!free -h

Train matrix: (111778, 182872)
Test matrix: (2952, 182872)
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.9Gi       7.3Gi       3.1Mi       2.8Gi       9.8Gi
Swap:             0B          0B          0B


In [49]:
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

matsnu_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

print("Training SVM...")
matsnu_svm.fit(
    matsnu_X_train,
    matsnu_train["label"]
)

print("Training complete.")

matsnu_pred = matsnu_svm.predict(matsnu_X_test)
matsnu_scores = matsnu_svm.decision_function(matsnu_X_test)

y_matsnu_test = matsnu_test_set["label"].values

accuracy = accuracy_score(y_matsnu_test, matsnu_pred)
precision = precision_score(y_matsnu_test, matsnu_pred)
recall = recall_score(y_matsnu_test, matsnu_pred)
f1 = f1_score(y_matsnu_test, matsnu_pred)
roc_auc = roc_auc_score(y_matsnu_test, matsnu_scores)

tn, fp, fn, tp = confusion_matrix(
    y_matsnu_test,
    matsnu_pred
).ravel()

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print("\n========== MATNSU-ONLY UNSEEN TEST ==========")
print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1       : {f1:.4f} ({f1*100:.2f}%)")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"FPR      : {fpr:.4f} ({fpr*100:.2f}%)")
print(f"FNR      : {fnr:.4f} ({fnr*100:.2f}%)")

print("\nConfusion Matrix:")
print(confusion_matrix(y_matsnu_test, matsnu_pred))

print(f"\nTN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")

print("\nRAM:")
!free -h

Training SVM...
Training complete.

========== MATNSU-ONLY UNSEEN TEST ==========
Accuracy : 0.5722 (57.22%)
Precision: 0.7950 (79.50%)
Recall   : 0.1944 (19.44%)
F1       : 0.3125 (31.25%)
ROC-AUC  : 0.7754
FPR      : 0.0501 (5.01%)
FNR      : 0.8056 (80.56%)

Confusion Matrix:
[[1402   74]
 [1189  287]]

TN = 1402
FP = 74
FN = 1189
TP = 287

RAM:
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.9Gi       7.3Gi       3.1Mi       2.8Gi       9.8Gi
Swap:             0B          0B          0B


In [50]:
import gc

gc.collect()

print("Garbage collection complete.")

Garbage collection complete.


In [51]:
# ============================================================
# STEP 1A — RAM-SAFE HYBRID EXPERIMENT SETUP
# ============================================================

import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Clean up anything Python can safely release
gc.collect()

print("unseen_train shape:", unseen_train.shape)
print("unseen_test shape :", unseen_test.shape)

# ------------------------------------------------------------
# 80/20 validation split from unseen_train
# IMPORTANT:
# unseen_test remains completely untouched until final evaluation.
# ------------------------------------------------------------

hybrid_train, hybrid_val = train_test_split(
    unseen_train,
    test_size=0.20,
    random_state=42,
    stratify=unseen_train["label"]
)

print("\nHybrid train:", hybrid_train.shape)
print("Hybrid val  :", hybrid_val.shape)
print("Unseen test :", unseen_test.shape)

print("\nClass distribution:")
print("Hybrid train:")
print(hybrid_train["label"].value_counts().sort_index())

print("\nHybrid validation:")
print(hybrid_val["label"].value_counts().sort_index())

print("\nUnseen test:")
print(unseen_test["label"].value_counts().sort_index())

gc.collect()

unseen_train shape: (109316, 7)
unseen_test shape : (10828, 7)

Hybrid train: (87452, 7)
Hybrid val  : (21864, 7)
Unseen test : (10828, 7)

Class distribution:
Hybrid train:
label
0    32360
1    55092
Name: count, dtype: int64

Hybrid validation:
label
0     8090
1    13774
Name: count, dtype: int64

Unseen test:
label
0    5414
1    5414
Name: count, dtype: int64


0

In [52]:
# ============================================================
# STEP 1B — CHARACTER MODEL FOR HYBRID EXPERIMENT
# RAM-SAFE VERSION
# ============================================================

import gc
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

gc.collect()

print("Starting character TF-IDF...")

# ------------------------------------------------------------
# Character TF-IDF
# ------------------------------------------------------------

char_vectorizer_h = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=3,
    max_features=200000,
    sublinear_tf=True,
    dtype=np.float32
)

char_X_train = char_vectorizer_h.fit_transform(
    hybrid_train["dga_string"].astype(str)
)

print("Train matrix:", char_X_train.shape)

char_X_val = char_vectorizer_h.transform(
    hybrid_val["dga_string"].astype(str)
)

print("Validation matrix:", char_X_val.shape)

char_X_test = char_vectorizer_h.transform(
    unseen_test["dga_string"].astype(str)
)

print("Test matrix:", char_X_test.shape)

# ------------------------------------------------------------
# Character SVM
# ------------------------------------------------------------

print("\nTraining LinearSVC...")

char_svm_h = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

char_svm_h.fit(
    char_X_train,
    hybrid_train["label"]
)

print("Character SVM training complete.")

# ------------------------------------------------------------
# IMPORTANT:
# Save ONLY scores.
# We don't need the huge sparse matrices after this.
# ------------------------------------------------------------

char_val_score = char_svm_h.decision_function(char_X_val)
char_test_score = char_svm_h.decision_function(char_X_test)

print("\nScores created:")
print("char_val_score :", char_val_score.shape)
print("char_test_score:", char_test_score.shape)

print("\nScore statistics:")
print("Validation mean:", char_val_score.mean())
print("Validation std :", char_val_score.std())
print("Test mean      :", char_test_score.mean())
print("Test std       :", char_test_score.std())

# ------------------------------------------------------------
# FREE MEMORY
# ------------------------------------------------------------

del char_X_train
del char_X_val
del char_X_test
del char_vectorizer_h
del char_svm_h

gc.collect()

print("\nCharacter model objects deleted.")
print("Only validation/test decision scores are retained.")

Starting character TF-IDF...
Train matrix: (87452, 133305)
Validation matrix: (21864, 133305)
Test matrix: (10828, 133305)

Training LinearSVC...
Character SVM training complete.

Scores created:
char_val_score : (21864,)
char_test_score: (10828,)

Score statistics:
Validation mean: 0.4285993329919813
Validation std : 1.2602325114783992
Test mean      : 0.05260864865889588
Test std       : 1.508067689683733

Character model objects deleted.
Only validation/test decision scores are retained.


In [53]:
# ============================================================
# STEP 1C — WORD-LEVEL MODEL FOR HYBRID EXPERIMENT
# RAM-SAFE VERSION
# ============================================================

import gc
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

gc.collect()

print("Starting word-level TF-IDF...")

# ------------------------------------------------------------
# Word TF-IDF
#
# Hyphens separate words, which is useful for domains such as:
# love-bear-shame
# radio-style-cake
#
# We use unigrams + bigrams.
# ------------------------------------------------------------

word_vectorizer_h = TfidfVectorizer(
    analyzer="word",
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2),
    min_df=2,
    max_features=40000,
    sublinear_tf=True,
    dtype=np.float32
)

word_X_train = word_vectorizer_h.fit_transform(
    hybrid_train["dga_string"].astype(str)
)

print("Train matrix:", word_X_train.shape)

word_X_val = word_vectorizer_h.transform(
    hybrid_val["dga_string"].astype(str)
)

print("Validation matrix:", word_X_val.shape)

word_X_test = word_vectorizer_h.transform(
    unseen_test["dga_string"].astype(str)
)

print("Test matrix:", word_X_test.shape)

# ------------------------------------------------------------
# Logistic Regression
#
# Gives us probabilities, which are useful for fusion.
# ------------------------------------------------------------

print("\nTraining LogisticRegression...")

word_lr_h = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    solver="liblinear",
    max_iter=100,
    random_state=42
)

word_lr_h.fit(
    word_X_train,
    hybrid_train["label"]
)

print("Word model training complete.")

# ------------------------------------------------------------
# Save ONLY probabilities
# ------------------------------------------------------------

word_val_prob = word_lr_h.predict_proba(word_X_val)[:, 1]
word_test_prob = word_lr_h.predict_proba(word_X_test)[:, 1]

print("\nProbabilities created:")
print("word_val_prob :", word_val_prob.shape)
print("word_test_prob:", word_test_prob.shape)

print("\nProbability statistics:")
print("Validation mean:", word_val_prob.mean())
print("Validation std :", word_val_prob.std())
print("Test mean      :", word_test_prob.mean())
print("Test std       :", word_test_prob.std())

# ------------------------------------------------------------
# FREE MEMORY
# ------------------------------------------------------------

del word_X_train
del word_X_val
del word_X_test
del word_vectorizer_h
del word_lr_h

gc.collect()

print("\nWord model objects deleted.")
print("Only validation/test probabilities are retained.")

Starting word-level TF-IDF...
Train matrix: (87452, 2475)
Validation matrix: (21864, 2475)
Test matrix: (10828, 2475)

Training LogisticRegression...
Word model training complete.

Probabilities created:
word_val_prob : (21864,)
word_test_prob: (10828,)

Probability statistics:
Validation mean: 0.511294471212655
Validation std : 0.07247064236223047
Test mean      : 0.5004509259518961
Test std       : 0.09049377304585293

Word model objects deleted.
Only validation/test probabilities are retained.


In [54]:
# ============================================================
# STEP 1D — VALIDATION-ONLY CHAR + WORD FUSION
# ============================================================
#
# IMPORTANT:
# We use ONLY hybrid_val here to choose:
#   1. fusion weight
#   2. decision threshold
#
# unseen_test is NOT used for tuning.
# ============================================================

import numpy as np
import pandas as pd

from scipy.special import logit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Convert word probabilities to logits
# ------------------------------------------------------------

word_val_logit = logit(
    np.clip(word_val_prob, 1e-6, 1 - 1e-6)
)

word_test_logit = logit(
    np.clip(word_test_prob, 1e-6, 1 - 1e-6)
)

# ------------------------------------------------------------
# 2. Normalize both model scores using VALIDATION statistics
#
# This puts the two models on roughly comparable scales.
# ------------------------------------------------------------

char_mean = char_val_score.mean()
char_std = char_val_score.std()

word_mean = word_val_logit.mean()
word_std = word_val_logit.std()

char_val_z = (
    (char_val_score - char_mean) / char_std
)

char_test_z = (
    (char_test_score - char_mean) / char_std
)

word_val_z = (
    (word_val_logit - word_mean) / word_std
)

word_test_z = (
    (word_test_logit - word_mean) / word_std
)

print("Validation normalization:")
print("Character mean:", char_mean)
print("Character std :", char_std)
print("Word logit mean:", word_mean)
print("Word logit std :", word_std)

# ------------------------------------------------------------
# 3. Fusion
#
# fused_score =
#     char_weight * character_score
#     +
#     (1 - char_weight) * word_score
#
# Higher score = more DGA-like.
# ------------------------------------------------------------

y_val = hybrid_val["label"].to_numpy()

results = []

weights = np.arange(0.0, 1.01, 0.05)

# Slightly finer threshold grid around the useful region
thresholds = np.arange(-1.50, 1.501, 0.05)

for char_weight in weights:

    word_weight = 1.0 - char_weight

    fused_val = (
        char_weight * char_val_z
        +
        word_weight * word_val_z
    )

    for threshold in thresholds:

        pred = (
            fused_val >= threshold
        ).astype(np.int8)

        tn, fp, fn, tp = confusion_matrix(
            y_val,
            pred,
            labels=[0, 1]
        ).ravel()

        accuracy = (tp + tn) / (tp + tn + fp + fn)

        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        fpr = (
            fp / (fp + tn)
            if (fp + tn) > 0
            else 0.0
        )

        fnr = (
            fn / (fn + tp)
            if (fn + tp) > 0
            else 0.0
        )

        results.append({
            "char_weight": char_weight,
            "word_weight": word_weight,
            "threshold": threshold,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "fpr": fpr,
            "fnr": fnr,
            "fp": fp,
            "fn": fn
        })

fusion_results = pd.DataFrame(results)

# ------------------------------------------------------------
# 4. Best F1
# ------------------------------------------------------------

best_f1 = (
    fusion_results
    .sort_values(
        ["f1", "recall", "precision"],
        ascending=False
    )
    .iloc[0]
)

print("\n" + "=" * 70)
print("BEST VALIDATION RESULT — F1")
print("=" * 70)

print(best_f1.to_string())

# ------------------------------------------------------------
# 5. Best FN-priority result
#
# Among configurations with FPR <= 5%, find the one
# with the highest recall.
# ------------------------------------------------------------

low_fpr = fusion_results[
    fusion_results["fpr"] <= 0.05
].copy()

if len(low_fpr) > 0:

    best_low_fpr = (
        low_fpr
        .sort_values(
            ["recall", "f1"],
            ascending=False
        )
        .iloc[0]
    )

    print("\n" + "=" * 70)
    print("BEST VALIDATION RESULT — RECALL WITH FPR <= 5%")
    print("=" * 70)

    print(best_low_fpr.to_string())

# ------------------------------------------------------------
# 6. Best F1 with recall >= 95%
# ------------------------------------------------------------

high_recall = fusion_results[
    fusion_results["recall"] >= 0.95
].copy()

if len(high_recall) > 0:

    best_high_recall = (
        high_recall
        .sort_values(
            ["f1", "fpr"],
            ascending=[False, True]
        )
        .iloc[0]
    )

    print("\n" + "=" * 70)
    print("BEST VALIDATION RESULT — F1 WITH RECALL >= 95%")
    print("=" * 70)

    print(best_high_recall.to_string())

# ------------------------------------------------------------
# 7. Show top 15 configurations
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 15 VALIDATION CONFIGURATIONS")
print("=" * 70)

display(
    fusion_results
    .sort_values(
        ["f1", "recall"],
        ascending=False
    )
    .head(15)
    [[
        "char_weight",
        "word_weight",
        "threshold",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "fpr",
        "fnr",
        "fp",
        "fn"
    ]]
    .reset_index(drop=True)
)

Validation normalization:
Character mean: 0.4285993329919813
Character std : 1.2602325114783992
Word logit mean: 0.024224882167796172
Word logit std : 0.4294844040593396

BEST VALIDATION RESULT — F1
char_weight      0.550000
word_weight      0.450000
threshold       -0.150000
accuracy         0.950832
precision        0.956503
recall           0.965878
f1               0.961168
fpr              0.074784
fnr              0.034122
fp             605.000000
fn             470.000000

BEST VALIDATION RESULT — RECALL WITH FPR <= 5%
char_weight      0.500000
word_weight      0.500000
threshold       -0.050000
accuracy         0.950192
precision        0.970198
recall           0.950123
f1               0.960056
fpr              0.049691
fnr              0.049877
fp             402.000000
fn             687.000000

BEST VALIDATION RESULT — F1 WITH RECALL >= 95%
char_weight      0.550000
word_weight      0.450000
threshold       -0.150000
accuracy         0.950832
precision        0.956503
rec

,char_weight,word_weight,threshold,accuracy,precision,recall,f1,fpr,fnr,fp,fn
0,0.55,0.45,-1.500000e-01,0.950832,0.956503,0.965878,0.961168,0.074784,0.034122,605,470
1,0.30,0.70,1.332268e-15,0.950604,0.955439,0.966676,0.961025,0.076761,0.033324,621,459
2,0.70,0.30,-2.500000e-01,0.950558,0.954718,0.967402,0.961018,0.078121,0.032598,632,449
3,0.60,0.40,-1.500000e-01,0.950832,0.961413,0.960505,0.960959,0.065637,0.039495,531,544
4,0.80,0.20,-3.000000e-01,0.950512,0.956350,0.965515,0.960910,0.075031,0.034485,607,475
5,0.95,0.05,-4.000000e-01,0.950421,0.955034,0.966822,0.960892,0.077503,0.033178,627,457
6,0.65,0.35,-2.000000e-01,0.950558,0.957933,0.963845,0.960880,0.072064,0.036155,583,498
7,0.15,0.85,1.000000e-01,0.950649,0.960397,0.961304,0.960850,0.067491,0.038696,546,533
8,0.45,0.55,-1.000000e-01,0.950284,0.953658,0.968128,0.960839,0.080099,0.031872,648,439
9,0.75,0.25,-2.500000e-01,0.950558,0.959126,0.962538,0.960829,0.069839,0.037462,565,516


In [55]:
# ============================================================
# STEP 1E — FINAL UNSEEN-FAMILY TEST
# ============================================================
#
# FROZEN configuration selected ONLY from validation:
#
#   Character weight = 0.90
#   Word weight      = 0.10
#   Threshold        = -0.10
#
# unseen_test has NOT been used for tuning.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# FROZEN SETTINGS
# ------------------------------------------------------------

CHAR_WEIGHT = 0.90
WORD_WEIGHT = 0.10
THRESHOLD = -0.10

y_test = unseen_test["label"].to_numpy()

# ------------------------------------------------------------
# Create final fused score
# ------------------------------------------------------------

hybrid_test_score = (
    CHAR_WEIGHT * char_test_z
    +
    WORD_WEIGHT * word_test_z
)

# ------------------------------------------------------------
# Final prediction
# ------------------------------------------------------------

hybrid_pred = (
    hybrid_test_score >= THRESHOLD
).astype(np.int8)

# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_test, hybrid_pred)

precision = precision_score(
    y_test,
    hybrid_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    hybrid_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    hybrid_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    hybrid_test_score
)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    hybrid_pred,
    labels=[0, 1]
).ravel()

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 70)
print("FINAL HYBRID RESULT — UNSEEN DGA FAMILIES")
print("=" * 70)

print(f"Character weight : {CHAR_WEIGHT:.2f}")
print(f"Word weight      : {WORD_WEIGHT:.2f}")
print(f"Threshold        : {THRESHOLD:.2f}")

print("\nOverall metrics:")
print(f"Accuracy         : {accuracy:.4%}")
print(f"Precision        : {precision:.4%}")
print(f"Recall           : {recall:.4%}")
print(f"F1               : {f1:.4%}")
print(f"ROC-AUC          : {roc_auc:.4%}")
print(f"FPR              : {fpr:.4%}")
print(f"FNR              : {fnr:.4%}")

print("\nConfusion matrix:")
print(f"TN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")

print("\nMatrix:")
print(
    confusion_matrix(
        y_test,
        hybrid_pred,
        labels=[0, 1]
    )
)

# ------------------------------------------------------------
# Save predictions for family-level analysis
# ------------------------------------------------------------

hybrid_test_results = unseen_test[
    ["domain", "dga_string", "label"]
].copy()

hybrid_test_results["hybrid_score"] = hybrid_test_score
hybrid_test_results["prediction"] = hybrid_pred

print("\nTest result dataframe:")
print(hybrid_test_results.shape)

FINAL HYBRID RESULT — UNSEEN DGA FAMILIES
Character weight : 0.90
Word weight      : 0.10
Threshold        : -0.10

Overall metrics:
Accuracy         : 83.3118%
Precision        : 99.0481%
Recall           : 67.2700%
F1               : 80.1232%
ROC-AUC          : 95.1802%
FPR              : 0.6465%
FNR              : 32.7300%

Confusion matrix:
TN = 5379
FP = 35
FN = 1772
TP = 3642

Matrix:
[[5379   35]
 [1772 3642]]

Test result dataframe:
(10828, 5)


In [56]:
# ============================================================
# STEP 1F — FAMILY-LEVEL HYBRID ANALYSIS
# CORRECTED VERSION
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Rebuild results INCLUDING family
# ------------------------------------------------------------

family_results = unseen_test[
    ["domain", "dga_string", "label", "family"]
].copy()

family_results["hybrid_score"] = hybrid_test_score
family_results["prediction"] = hybrid_pred

# ------------------------------------------------------------
# Add result category
# ------------------------------------------------------------

family_results["result"] = np.select(
    [
        (family_results["label"] == 1) &
        (family_results["prediction"] == 1),

        (family_results["label"] == 1) &
        (family_results["prediction"] == 0),

        (family_results["label"] == 0) &
        (family_results["prediction"] == 1),

        (family_results["label"] == 0) &
        (family_results["prediction"] == 0)
    ],
    [
        "TP",
        "FN",
        "FP",
        "TN"
    ],
    default="UNKNOWN"
)

# ------------------------------------------------------------
# DGA family recall
# ------------------------------------------------------------

dga_test = family_results[
    family_results["label"] == 1
].copy()

family_summary = (
    dga_test
    .groupby("family", dropna=False)
    .agg(
        total_dga=("label", "size"),
        detected=("prediction", "sum")
    )
)

family_summary["missed"] = (
    family_summary["total_dga"]
    - family_summary["detected"]
)

family_summary["recall"] = (
    family_summary["detected"]
    / family_summary["total_dga"]
)

family_summary = family_summary.sort_values(
    "recall"
)

print("=" * 70)
print("HYBRID — DGA FAMILY RECALL")
print("=" * 70)

display(
    family_summary.style.format({
        "recall": "{:.2%}"
    })
)

# ------------------------------------------------------------
# Explicit results for the three unseen families
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("KEY UNSEEN FAMILIES")
print("=" * 70)

for family in ["bamital", "goz", "matsnu"]:

    if family in family_summary.index:

        row = family_summary.loc[family]

        print(
            f"{family:10s} | "
            f"total={int(row['total_dga']):5d} | "
            f"detected={int(row['detected']):5d} | "
            f"missed={int(row['missed']):5d} | "
            f"recall={row['recall']:.2%}"
        )

# ------------------------------------------------------------
# Legitimate false positives
# ------------------------------------------------------------

legit_test = family_results[
    family_results["label"] == 0
]

fp_count = (
    legit_test["prediction"] == 1
).sum()

print("\n" + "=" * 70)
print("LEGITIMATE FALSE POSITIVES")
print("=" * 70)

print(f"Legitimate domains : {len(legit_test)}")
print(f"False positives    : {fp_count}")
print(
    f"FPR                : "
    f"{fp_count / len(legit_test):.2%}"
)

HYBRID — DGA FAMILY RECALL


,total_dga,detected,missed,recall
family,,,,
matsnu,1476,116,1360,7.86%
bamital,971,560,411,57.67%
goz,2967,2966,1,99.97%



KEY UNSEEN FAMILIES
bamital    | total=  971 | detected=  560 | missed=  411 | recall=57.67%
goz        | total= 2967 | detected= 2966 | missed=    1 | recall=99.97%
matsnu     | total= 1476 | detected=  116 | missed= 1360 | recall=7.86%

LEGITIMATE FALSE POSITIVES
Legitimate domains : 5414
False positives    : 35
FPR                : 0.65%


In [57]:
import gc
gc.collect()

print("Ready for few-shot experiment.")
print("unseen_train:", unseen_train.shape)
print("unseen_test :", unseen_test.shape)

Ready for few-shot experiment.
unseen_train: (109316, 7)
unseen_test : (10828, 7)


In [58]:
# ============================================================
# STEP 2A — RECOVER MATSNU FROM final_dataset
# ============================================================

import gc

gc.collect()

print("final_dataset shape:", final_dataset.shape)

print("\nColumns:")
print(final_dataset.columns.tolist())

# ------------------------------------------------------------
# Find Matsnu in the original dataset
# ------------------------------------------------------------

matsnu_pool = final_dataset[
    (final_dataset["label"] == 1) &
    (final_dataset["family"] == "matsnu")
].copy()

print("\nMatsnu pool:", matsnu_pool.shape)

print("\nMatsnu count:")
print(len(matsnu_pool))

print("\nExample Matsnu domains:")
print(
    matsnu_pool[
        ["domain", "dga_string", "label", "family"]
    ].head(10)
)

gc.collect()

final_dataset shape: (114730, 7)

Columns:
['domain', 'dga_string', 'label', 'source', 'family', 'length', 'length_group']

Matsnu pool: (1476, 7)

Matsnu count:
1476

Example Matsnu domains:
                                  domain                       dga_string  \
52          hopetermstaffforceinvite.com         hopetermstaffforceinvite   
57      impressionwishmachinerespond.com     impressionwishmachinerespond   
66   failuremistakesmoketoolexercise.com  failuremistakesmoketoolexercise   
171        homespringweighlawyerpage.com        homespringweighlawyerpage   
233     perceptionstrokefeaturebirth.com     perceptionstrokefeaturebirth   
274       trainingdistancefirmstring.com       trainingdistancefirmstring   
307      riverpartstringslicestretch.com      riverpartstringslicestretch   
308        yeararecreamaddressstrain.com        yeararecreamaddressstrain   
381      concertstudychangeprintrock.com      concertstudychangeprintrock   
384      medicinepagebelongrentcable.c

0

In [59]:
# ============================================================
# STEP 2B — VALID 5% MATSNU FEW-SHOT EXPERIMENT
# ============================================================

import gc
import pandas as pd

gc.collect()

# ------------------------------------------------------------
# 1. Get Matsnu from the existing unseen test
# ------------------------------------------------------------

matsnu_test_pool = unseen_test[
    unseen_test["family"] == "matsnu"
].copy()

legit_test_pool = unseen_test[
    unseen_test["label"] == 0
].copy()

print("Matsnu pool:", len(matsnu_test_pool))
print("Legitimate pool:", len(legit_test_pool))

# ------------------------------------------------------------
# 2. Take 5% Matsnu for few-shot training
# ------------------------------------------------------------

n_matsnu_train = int(round(len(matsnu_test_pool) * 0.05))

matsnu_exposure_5 = matsnu_test_pool.sample(
    n=n_matsnu_train,
    random_state=42
).copy()

# Remaining Matsnu = evaluation
matsnu_eval_5 = matsnu_test_pool.drop(
    matsnu_exposure_5.index
).copy()

print("\nMatsnu exposed to training:", len(matsnu_exposure_5))
print("Matsnu remaining for evaluation:", len(matsnu_eval_5))

# ------------------------------------------------------------
# 3. Build balanced legitimate evaluation set
# ------------------------------------------------------------

legit_eval_5 = legit_test_pool.sample(
    n=len(matsnu_eval_5),
    random_state=42
).copy()

# ------------------------------------------------------------
# 4. Build few-shot evaluation set
# ------------------------------------------------------------

fewshot_test_5 = pd.concat(
    [
        matsnu_eval_5,
        legit_eval_5
    ],
    ignore_index=True
)

fewshot_test_5 = fewshot_test_5.sample(
    frac=1.0,
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Build few-shot training set
#
# unseen_train contains all non-Matsnu training data.
# Add the 675 Matsnu examples.
# ------------------------------------------------------------

fewshot_train_5 = pd.concat(
    [
        unseen_train,
        matsnu_exposure_5
    ],
    ignore_index=True
)

fewshot_train_5 = fewshot_train_5.sample(
    frac=1.0,
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# 6. Verify
# ------------------------------------------------------------

print("\n==============================")
print("5% FEW-SHOT DATASET")
print("==============================")

print("\nTraining:", fewshot_train_5.shape)
print("Test    :", fewshot_test_5.shape)

print("\nTraining class distribution:")
print(
    fewshot_train_5["label"]
    .value_counts()
    .sort_index()
)

print("\nTest class distribution:")
print(
    fewshot_test_5["label"]
    .value_counts()
    .sort_index()
)

print("\nMatsnu in training:",
      (fewshot_train_5["family"] == "matsnu").sum())

print("Matsnu in test:",
      (fewshot_test_5["family"] == "matsnu").sum())

# ------------------------------------------------------------
# 7. Verify train/test domain separation
# ------------------------------------------------------------

train_domains = set(
    fewshot_train_5["domain"].astype(str)
)

test_domains_5 = set(
    fewshot_test_5["domain"].astype(str)
)

overlap = train_domains & test_domains_5

print("\nTrain/test domain overlap:", len(overlap))

if len(overlap) == 0:
    print("✓ No domain leakage.")
else:
    print("⚠ WARNING: domain leakage detected!")

gc.collect()

Matsnu pool: 1476
Legitimate pool: 5414

Matsnu exposed to training: 74
Matsnu remaining for evaluation: 1402

5% FEW-SHOT DATASET

Training: (109390, 7)
Test    : (2804, 7)

Training class distribution:
label
0    40450
1    68940
Name: count, dtype: int64

Test class distribution:
label
0    1402
1    1402
Name: count, dtype: int64

Matsnu in training: 74
Matsnu in test: 1402

Train/test domain overlap: 1402
⚠ WARNING: domain leakage detected!


0

In [60]:
# ============================================================
# STEP 2C — ENFORCE STRICT DOMAIN-LEVEL SEPARATION
# ============================================================

import gc

gc.collect()

# Domains that appear in the new few-shot test set
fewshot_test_domains = set(
    fewshot_test_5["domain"].astype(str)
)

print("Unique domains in few-shot test:",
      len(fewshot_test_domains))

# ------------------------------------------------------------
# Remove ANY matching domain from the original unseen_train
# ------------------------------------------------------------

base_train_clean = unseen_train[
    ~unseen_train["domain"].astype(str).isin(fewshot_test_domains)
].copy()

print("\nOriginal unseen_train:", len(unseen_train))
print("After removing test-domain duplicates:", len(base_train_clean))
print("Rows removed:",
      len(unseen_train) - len(base_train_clean))

# ------------------------------------------------------------
# Add the 675 Matsnu exposure examples
# ------------------------------------------------------------

fewshot_train_5 = pd.concat(
    [
        base_train_clean,
        matsnu_exposure_5
    ],
    ignore_index=True
)

fewshot_train_5 = fewshot_train_5.sample(
    frac=1.0,
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# Verify strict domain-level separation
# ------------------------------------------------------------

train_domains = set(
    fewshot_train_5["domain"].astype(str)
)

test_domains = set(
    fewshot_test_5["domain"].astype(str)
)

overlap = train_domains & test_domains

print("\n==============================")
print("STRICT LEAKAGE CHECK")
print("==============================")

print("Training:", fewshot_train_5.shape)
print("Test:", fewshot_test_5.shape)

print("Train/test domain overlap:", len(overlap))

if len(overlap) == 0:
    print("✓ PERFECT: zero domain overlap.")
else:
    print("⚠ WARNING: overlap still exists.")

# ------------------------------------------------------------
# Check Matsnu exposure
# ------------------------------------------------------------

print("\nMatsnu in training:",
      (fewshot_train_5["family"] == "matsnu").sum())

print("Matsnu in test:",
      (fewshot_test_5["family"] == "matsnu").sum())

print("\nTraining class distribution:")
print(
    fewshot_train_5["label"]
    .value_counts()
    .sort_index()
)

print("\nTest class distribution:")
print(
    fewshot_test_5["label"]
    .value_counts()
    .sort_index()
)

gc.collect()

Unique domains in few-shot test: 2804

Original unseen_train: 109316
After removing test-domain duplicates: 107914
Rows removed: 1402

STRICT LEAKAGE CHECK
Training: (107988, 7)
Test: (2804, 7)
Train/test domain overlap: 0
✓ PERFECT: zero domain overlap.

Matsnu in training: 74
Matsnu in test: 1402

Training class distribution:
label
0    39048
1    68940
Name: count, dtype: int64

Test class distribution:
label
0    1402
1    1402
Name: count, dtype: int64


0

In [61]:
# ============================================================
# STEP 2D — 5% MATSNU FEW-SHOT CHARACTER SVM
# ============================================================

import gc
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

gc.collect()

# ------------------------------------------------------------
# Prepare text
# ------------------------------------------------------------

fewshot_train_text = (
    fewshot_train_5["dga_string"]
    .astype(str)
)

fewshot_test_text = (
    fewshot_test_5["dga_string"]
    .astype(str)
)

y_fewshot_train = fewshot_train_5["label"].values
y_fewshot_test = fewshot_test_5["label"].values

print("Training samples:", len(fewshot_train_text))
print("Test samples:", len(fewshot_test_text))

# ------------------------------------------------------------
# Character TF-IDF
# ------------------------------------------------------------

fewshot_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=3,
    max_features=200000,
    sublinear_tf=True,
    dtype=np.float32
)

print("\nFitting TF-IDF...")

X_fewshot_train = fewshot_vectorizer.fit_transform(
    fewshot_train_text
)

print("Training matrix:", X_fewshot_train.shape)

X_fewshot_test = fewshot_vectorizer.transform(
    fewshot_test_text
)

print("Test matrix:", X_fewshot_test.shape)

# ------------------------------------------------------------
# Train Linear SVM
# ------------------------------------------------------------

print("\nTraining Linear SVM...")

fewshot_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

fewshot_svm.fit(
    X_fewshot_train,
    y_fewshot_train
)

print("Training complete.")

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

fewshot_pred = fewshot_svm.predict(
    X_fewshot_test
)

fewshot_score = fewshot_svm.decision_function(
    X_fewshot_test
)

# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_fewshot_test,
    fewshot_pred
)

precision = precision_score(
    y_fewshot_test,
    fewshot_pred
)

recall = recall_score(
    y_fewshot_test,
    fewshot_pred
)

f1 = f1_score(
    y_fewshot_test,
    fewshot_pred
)

roc_auc = roc_auc_score(
    y_fewshot_test,
    fewshot_score
)

cm = confusion_matrix(
    y_fewshot_test,
    fewshot_pred
)

tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print("\n==============================")
print("5% FEW-SHOT RESULTS")
print("==============================")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"FPR      : {fpr:.4f}")
print(f"FNR      : {fnr:.4f}")

print("\nConfusion matrix:")
print(cm)

# ------------------------------------------------------------
# Matsnu-specific recall
# ------------------------------------------------------------

matsnu_mask = (
    fewshot_test_5["family"].values == "matsnu"
)

matsnu_true = y_fewshot_test[matsnu_mask]
matsnu_pred = fewshot_pred[matsnu_mask]

matsnu_recall = recall_score(
    matsnu_true,
    matsnu_pred
)

print("\n==============================")
print("MATSNU-SPECIFIC RESULT")
print("==============================")

print("Matsnu test samples:", matsnu_mask.sum())
print(f"Matsnu recall: {matsnu_recall:.4f}")

# ------------------------------------------------------------
# Legitimate FPR
# ------------------------------------------------------------

legit_mask = (
    fewshot_test_5["label"].values == 0
)

legit_true = y_fewshot_test[legit_mask]
legit_pred = fewshot_pred[legit_mask]

legit_fp = np.sum(legit_pred == 1)
legit_total = len(legit_true)

legit_fpr = legit_fp / legit_total

print(f"Legitimate FPR: {legit_fpr:.4f}")

gc.collect()

Training samples: 107988
Test samples: 2804

Fitting TF-IDF...
Training matrix: (107988, 167011)
Test matrix: (2804, 167011)

Training Linear SVM...
Training complete.

5% FEW-SHOT RESULTS
Accuracy : 0.6002
Precision: 0.8022
Recall   : 0.2660
F1       : 0.3996
ROC-AUC  : 0.8059
FPR      : 0.0656
FNR      : 0.7340

Confusion matrix:
[[1310   92]
 [1029  373]]

MATSNU-SPECIFIC RESULT
Matsnu test samples: 1402
Matsnu recall: 0.2660
Legitimate FPR: 0.0656


0

In [62]:
# ============================================================
# STEP 2E — 5% FEW-SHOT THRESHOLD ANALYSIS
# ============================================================

import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

thresholds = np.arange(-2.0, 1.01, 0.10)

results_5 = []

for threshold in thresholds:

    pred = (fewshot_score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_fewshot_test,
        pred
    ).ravel()

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    results_5.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_fewshot_test, pred),
        "precision": precision_score(y_fewshot_test, pred, zero_division=0),
        "recall": recall_score(y_fewshot_test, pred),
        "f1": f1_score(y_fewshot_test, pred),
        "fpr": fpr,
        "fnr": fnr,
        "fp": fp,
        "fn": fn
    })

threshold_df_5 = pd.DataFrame(results_5)

print("Top thresholds by F1:")
print(
    threshold_df_5
    .sort_values("f1", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\nBest thresholds with FPR <= 5%:")
print(
    threshold_df_5[
        threshold_df_5["fpr"] <= 0.05
    ]
    .sort_values("recall", ascending=False)
    .head(10)
    .to_string(index=False)
)

# ------------------------------------------------------------
# Matsnu recall specifically
# ------------------------------------------------------------

matsnu_mask_5 = (
    fewshot_test_5["family"].values == "matsnu"
)

matsnu_threshold_results = []

for threshold in thresholds:

    pred = (fewshot_score >= threshold).astype(int)

    matsnu_recall = recall_score(
        y_fewshot_test[matsnu_mask_5],
        pred[matsnu_mask_5]
    )

    legit_mask = (
        y_fewshot_test == 0
    )

    legit_fpr = np.mean(
        pred[legit_mask] == 1
    )

    matsnu_threshold_results.append({
        "threshold": threshold,
        "matsnu_recall": matsnu_recall,
        "legit_fpr": legit_fpr
    })

matsnu_threshold_df_5 = pd.DataFrame(
    matsnu_threshold_results
)

print("\nMatsnu recall vs threshold:")
print(
    matsnu_threshold_df_5.to_string(index=False)
)

Top thresholds by F1:
 threshold  accuracy  precision   recall       f1      fpr      fnr   fp  fn
      -0.9  0.732525   0.682941 0.868046 0.764447 0.402996 0.131954  565 185
      -0.8  0.744650   0.711207 0.823823 0.763384 0.334522 0.176177  469 247
      -1.0  0.716833   0.656701 0.908702 0.762418 0.475036 0.091298  666 128
      -1.1  0.697218   0.634028 0.932953 0.754978 0.538516 0.067047  755  94
      -0.7  0.740014   0.733195 0.754636 0.743761 0.274608 0.245364  385 344
      -1.2  0.667974   0.606706 0.955064 0.742034 0.619116 0.044936  868  63
      -1.3  0.638017   0.582940 0.970043 0.728246 0.694009 0.029957  973  42
      -0.6  0.732168   0.753307 0.690442 0.720506 0.226106 0.309558  317 434
      -1.4  0.607347   0.561554 0.979315 0.713803 0.764622 0.020685 1072  29
      -1.5  0.583452   0.546063 0.989301 0.703704 0.822397 0.010699 1153  15

Best thresholds with FPR <= 5%:
 threshold  accuracy  precision   recall       f1      fpr      fnr  fp   fn
       0.1  0.579173 

In [63]:
# ============================================================
# CANDIDATE MODEL LEADERBOARD
# Sequential training + aggressive RAM cleanup
#
# Evaluation set:
#   unseen_test
#   - 21,721 legitimate
#   - 21,721 unseen-family DGA
#   - DGA families: Bamital, GOZ, Matsnu
#
# NOTE:
# The 5% few-shot experiment is NOT used here.
# This leaderboard is for the original unseen-family benchmark.
# ============================================================

import gc
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

gc.collect()

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

MAX_FEATURES_CHAR = 200000
MAX_FEATURES_WORD = 40000

RANDOM_STATE = 42

# ------------------------------------------------------------
# Prepare common evaluation data
# ------------------------------------------------------------

eval_text = unseen_test["dga_string"].astype(str).values
y_eval = unseen_test["label"].astype(np.int8).values
eval_family = unseen_test["family"].astype(str).values

print("Evaluation set:", len(eval_text))
print("Legitimate:", int((y_eval == 0).sum()))
print("DGA:", int((y_eval == 1).sum()))

print("\nDGA family distribution:")
print(
    pd.Series(
        eval_family[y_eval == 1]
    ).value_counts()
)

# ------------------------------------------------------------
# Helper: evaluate predictions
# ------------------------------------------------------------

leaderboard = []
score_store = {}

def evaluate_model(
    model_name,
    y_true,
    predictions,
    scores,
    family_values
):

    cm = confusion_matrix(
        y_true,
        predictions
    )

    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    result = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_true,
            scores
        ),
        "fpr": fpr,
        "fnr": fnr,
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "tn": int(tn)
    }

    # --------------------------------------------------------
    # Family-specific recall
    # --------------------------------------------------------

    for family in ["bamital", "goz", "matsnu"]:

        mask = (
            (family_values == family) &
            (y_true == 1)
        )

        if mask.sum() > 0:
            result[f"{family}_recall"] = recall_score(
                y_true[mask],
                predictions[mask],
                zero_division=0
            )
        else:
            result[f"{family}_recall"] = np.nan

    leaderboard.append(result)

    print("\n" + "=" * 65)
    print(model_name)
    print("=" * 65)

    print(f"Accuracy : {result['accuracy']:.4f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall   : {result['recall']:.4f}")
    print(f"F1       : {result['f1']:.4f}")
    print(f"ROC-AUC  : {result['roc_auc']:.4f}")
    print(f"FPR      : {result['fpr']:.4f}")
    print(f"FNR      : {result['fnr']:.4f}")

    print("\nFamily recall:")
    print(
        f"  Bamital: {result['bamital_recall']:.4f}"
    )
    print(
        f"  GOZ    : {result['goz_recall']:.4f}"
    )
    print(
        f"  Matsnu : {result['matsnu_recall']:.4f}"
    )

    print("\nConfusion matrix:")
    print(cm)


# ============================================================
# MODEL 1 — FROZEN CHAR SVM 2–5
# ============================================================

print("\n\n")
print("#" * 65)
print("# MODEL 1 — FROZEN CHAR SVM 2–5")
print("#" * 65)

import joblib

BASE_MODEL_PATH = "/content/models/char_svm_model.joblib"
BASE_VECTORIZER_PATH = "/content/models/char_tfidf_vectorizer.joblib"

if not os.path.exists(BASE_MODEL_PATH):
    raise FileNotFoundError(
        f"Missing model: {BASE_MODEL_PATH}"
    )

if not os.path.exists(BASE_VECTORIZER_PATH):
    raise FileNotFoundError(
        f"Missing vectorizer: {BASE_VECTORIZER_PATH}"
    )

base_vectorizer = joblib.load(
    BASE_VECTORIZER_PATH
)

base_model = joblib.load(
    BASE_MODEL_PATH
)

print("Transforming unseen test...")

X_eval = base_vectorizer.transform(
    eval_text
)

print("Matrix:", X_eval.shape)

base_scores = base_model.decision_function(
    X_eval
)

base_pred = (
    base_scores >= 0
).astype(np.int8)

evaluate_model(
    "Char SVM 2-5 (Frozen Baseline)",
    y_eval,
    base_pred,
    base_scores,
    eval_family
)

# Store only scores, not matrix/model
score_store["char_2_5"] = base_scores.astype(
    np.float32
)

del X_eval
del base_scores
del base_pred
del base_model
del base_vectorizer

gc.collect()

print("\nRAM cleaned after Model 1.")


# ============================================================
# MODEL 2 — CHAR SVM 2–6
# ============================================================

print("\n\n")
print("#" * 65)
print("# MODEL 2 — CHAR SVM 2–6")
print("#" * 65)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

char26_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 6),
    min_df=3,
    max_features=MAX_FEATURES_CHAR,
    sublinear_tf=True,
    dtype=np.float32
)

print("Fitting 2–6 TF-IDF...")

X_train_char26 = char26_vectorizer.fit_transform(
    unseen_train["dga_string"].astype(str)
)

print("Training matrix:", X_train_char26.shape)

char26_model = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

print("Training SVM...")

char26_model.fit(
    X_train_char26,
    unseen_train["label"].values
)

print("Transforming test...")

X_eval_char26 = char26_vectorizer.transform(
    eval_text
)

char26_scores = char26_model.decision_function(
    X_eval_char26
)

char26_pred = (
    char26_scores >= 0
).astype(np.int8)

evaluate_model(
    "Char SVM 2-6",
    y_eval,
    char26_pred,
    char26_scores,
    eval_family
)

score_store["char_2_6"] = char26_scores.astype(
    np.float32
)

# Aggressive cleanup
del X_train_char26
del X_eval_char26
del char26_model
del char26_vectorizer
del char26_scores
del char26_pred

gc.collect()

print("\nRAM cleaned after Model 2.")


# ============================================================
# MODEL 3 — CHAR SVM 3–5
# ============================================================

print("\n\n")
print("#" * 65)
print("# MODEL 3 — CHAR SVM 3–5")
print("#" * 65)

char35_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=3,
    max_features=MAX_FEATURES_CHAR,
    sublinear_tf=True,
    dtype=np.float32
)

print("Fitting 3–5 TF-IDF...")

X_train_char35 = char35_vectorizer.fit_transform(
    unseen_train["dga_string"].astype(str)
)

print("Training matrix:", X_train_char35.shape)

char35_model = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

print("Training SVM...")

char35_model.fit(
    X_train_char35,
    unseen_train["label"].values
)

print("Transforming test...")

X_eval_char35 = char35_vectorizer.transform(
    eval_text
)

char35_scores = char35_model.decision_function(
    X_eval_char35
)

char35_pred = (
    char35_scores >= 0
).astype(np.int8)

evaluate_model(
    "Char SVM 3-5",
    y_eval,
    char35_pred,
    char35_scores,
    eval_family
)

score_store["char_3_5"] = char35_scores.astype(
    np.float32
)

# Aggressive cleanup
del X_train_char35
del X_eval_char35
del char35_model
del char35_vectorizer
del char35_scores
del char35_pred

gc.collect()

print("\nRAM cleaned after Model 3.")


#################################################################
# MODEL 4 — RANDOM FOREST
#################################################################

import gc
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Preparing RF test features...")

# ---------------------------------------------------------------
# 1. RF feature list
# ---------------------------------------------------------------

rf_features = [
    "domain_length",
    "num_digits",
    "digit_ratio",
    "num_letters",
    "num_vowels",
    "num_consonants",
    "vowel_ratio",
    "consonant_ratio",
    "num_special_chars",
    "special_char_ratio",
    "hyphen_count",
    "dot_count",
    "subdomain_count",
    "unique_chars",
    "unique_char_ratio",
    "shannon_entropy",
    "max_consecutive_digits",
    "max_consecutive_consonants",
    "character_transitions",
    "repeated_chars",
    "repeated_ratio",
    "max_consecutive_vowels",
    "letter_digit_transitions",
    "digit_letter_transitions"
]

# ---------------------------------------------------------------
# 2. Build one feature lookup from features_df
# ---------------------------------------------------------------

rf_lookup = (
    features_df[
        ["domain"] + rf_features
    ]
    .drop_duplicates(subset=["domain"])
    .set_index("domain")
)

# ---------------------------------------------------------------
# 3. Get training features
# ---------------------------------------------------------------

X_rf_train = rf_lookup.loc[
    unseen_train["domain"],
    rf_features
].astype(np.float32)

y_rf_train = unseen_train["label"].astype(np.int8).values

# ---------------------------------------------------------------
# 4. Get test features
# ---------------------------------------------------------------

X_rf_test = rf_lookup.loc[
    unseen_test["domain"],
    rf_features
].astype(np.float32)

y_rf_test = unseen_test["label"].astype(np.int8).values

print("RF training matrix:", X_rf_train.shape)
print("RF test matrix    :", X_rf_test.shape)

# ---------------------------------------------------------------
# 5. Train Random Forest
# ---------------------------------------------------------------

print("Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_rf_train, y_rf_train)

# ---------------------------------------------------------------
# 6. Predict
# ---------------------------------------------------------------

print("Predicting...")

rf_pred = rf_model.predict(X_rf_test)
rf_score = rf_model.predict_proba(X_rf_test)[:, 1]

# ---------------------------------------------------------------
# 7. Metrics
# ---------------------------------------------------------------

rf_acc = accuracy_score(y_rf_test, rf_pred)
rf_prec = precision_score(y_rf_test, rf_pred)
rf_rec = recall_score(y_rf_test, rf_pred)
rf_f1 = f1_score(y_rf_test, rf_pred)
rf_auc = roc_auc_score(y_rf_test, rf_score)

rf_cm = confusion_matrix(y_rf_test, rf_pred)

tn, fp, fn, tp = rf_cm.ravel()

rf_fpr = fp / (fp + tn)
rf_fnr = fn / (fn + tp)

print("\n=================================================================")
print("Random Forest — 24 Structural Features")
print("=================================================================")

print(f"Accuracy : {rf_acc:.4f}")
print(f"Precision: {rf_prec:.4f}")
print(f"Recall   : {rf_rec:.4f}")
print(f"F1       : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_auc:.4f}")
print(f"FPR      : {rf_fpr:.4f}")
print(f"FNR      : {rf_fnr:.4f}")

# ---------------------------------------------------------------
# 8. Family recall
# ---------------------------------------------------------------

rf_family_df = unseen_test[
    ["family", "label"]
].copy()

rf_family_df["prediction"] = rf_pred

print("\nFamily recall:")

for family in ["bamital", "goz", "matsnu"]:

    subset = rf_family_df[
        (rf_family_df["family"] == family) &
        (rf_family_df["label"] == 1)
    ]

    family_recall = subset["prediction"].mean()

    print(f"  {family.capitalize():7s}: {family_recall:.4f}")

print("\nConfusion matrix:")
print(rf_cm)

# ---------------------------------------------------------------
# 9. Save ONLY scores for future ensemble
# ---------------------------------------------------------------

score_store["random_forest"] = rf_score.astype(np.float32)

# ---------------------------------------------------------------
# 10. Aggressive RAM cleanup
# ---------------------------------------------------------------

del (
    rf_lookup,
    X_rf_train,
    y_rf_train,
    X_rf_test,
    y_rf_test,
    rf_pred,
    rf_score,
    rf_family_df,
    rf_model
)

gc.collect()

print("\nRAM cleaned after Model 4.")

# ============================================================
# MODEL 5 — WORD TF-IDF + LOGISTIC REGRESSION
# ============================================================

print("\n\n")
print("#" * 65)
print("# MODEL 5 — WORD TF-IDF + LOGISTIC REGRESSION")
print("#" * 65)

from sklearn.linear_model import LogisticRegression

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2),
    min_df=2,
    max_features=MAX_FEATURES_WORD,
    sublinear_tf=True,
    dtype=np.float32
)

print("Fitting word TF-IDF...")

X_train_word = word_vectorizer.fit_transform(
    unseen_train["dga_string"].astype(str)
)

print("Training matrix:", X_train_word.shape)

word_model = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    solver="liblinear",
    max_iter=100,
    random_state=RANDOM_STATE
)

print("Training Logistic Regression...")

word_model.fit(
    X_train_word,
    unseen_train["label"].values
)

print("Transforming test...")

X_eval_word = word_vectorizer.transform(
    eval_text
)

word_prob = word_model.predict_proba(
    X_eval_word
)[:, 1]

word_pred = (
    word_prob >= 0.5
).astype(np.int8)

evaluate_model(
    "Word TF-IDF + Logistic Regression",
    y_eval,
    word_pred,
    word_prob,
    eval_family
)

score_store["word_lr"] = word_prob.astype(
    np.float32
)

# Aggressive cleanup
del X_train_word
del X_eval_word
del word_model
del word_vectorizer
del word_prob
del word_pred

gc.collect()

print("\nRAM cleaned after Model 5.")


# ============================================================
# FINAL LEADERBOARD
# ============================================================

leaderboard_df = pd.DataFrame(
    leaderboard
).sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

print("\n\n")
print("=" * 100)
print("FINAL CANDIDATE MODEL LEADERBOARD")
print("=" * 100)

display_columns = [
    "model",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "fpr",
    "fnr",
    "bamital_recall",
    "goz_recall",
    "matsnu_recall"
]

print(
    leaderboard_df[
        display_columns
    ].to_string(index=False)
)

print("\n")
print("Best overall F1:")
print(
    leaderboard_df.iloc[0]["model"]
)

print("\nBest Matsnu recall:")
best_matsnu_idx = leaderboard_df[
    "matsnu_recall"
].idxmax()

print(
    leaderboard_df.loc[
        best_matsnu_idx,
        "model"
    ]
)

print("\nBest Bamital recall:")
best_bamital_idx = leaderboard_df[
    "bamital_recall"
].idxmax()

print(
    leaderboard_df.loc[
        best_bamital_idx,
        "model"
    ]
)

print("\nScore arrays retained for future ensemble:")
print(
    list(score_store.keys())
)

gc.collect()

print("\n✓ Candidate model evaluation complete.")
print("✓ Temporary TF-IDF matrices/models deleted.")

Evaluation set: 10828
Legitimate: 5414
DGA: 5414

DGA family distribution:
goz        2967
matsnu     1476
bamital     971
Name: count, dtype: int64



#################################################################
# MODEL 1 — FROZEN CHAR SVM 2–5
#################################################################
Transforming unseen test...
Matrix: (10828, 148627)

Char SVM 2-5 (Frozen Baseline)
Accuracy : 0.9901
Precision: 0.9985
Recall   : 0.9817
F1       : 0.9900
ROC-AUC  : 0.9993
FPR      : 0.0015
FNR      : 0.0183

Family recall:
  Bamital: 1.0000
  GOZ    : 1.0000
  Matsnu : 0.9329

Confusion matrix:
[[5406    8]
 [  99 5315]]

RAM cleaned after Model 1.



#################################################################
# MODEL 2 — CHAR SVM 2–6
#################################################################
Fitting 2–6 TF-IDF...
Training matrix: (109316, 185585)
Training SVM...
Transforming test...

Char SVM 2-6
Accuracy : 0.8738
Precision: 0.9980
Recall   : 0.7490
F1       

In [64]:
# ============================================================
# MODEL 4 — RANDOM FOREST
# Correct domain-based feature alignment
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
import numpy as np

rf_features = feature_columns

print("Preparing Random Forest features...")

# Build domain -> feature lookup
rf_lookup = (
    features_df[["domain"] + rf_features]
    .drop_duplicates(subset=["domain"])
    .set_index("domain")
)

# Align features using DOMAIN, not dataframe index
X_rf_train = rf_lookup.loc[
    unseen_train["domain"],
    rf_features
].astype(np.float32)

X_rf_test = rf_lookup.loc[
    unseen_test["domain"],
    rf_features
].astype(np.float32)

y_rf_train = unseen_train["label"].astype(int)
y_rf_test = unseen_test["label"].astype(int)

print("Training shape:", X_rf_train.shape)
print("Test shape:", X_rf_test.shape)

# ------------------------------------------------------------
# Train Random Forest
# ------------------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\nTraining Random Forest...")

rf_model.fit(X_rf_train, y_rf_train)

print("Training complete.")

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

rf_pred = rf_model.predict(X_rf_test)
rf_prob = rf_model.predict_proba(X_rf_test)[:, 1]

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_rf_test, rf_pred)
precision = precision_score(y_rf_test, rf_pred)
recall = recall_score(y_rf_test, rf_pred)
f1 = f1_score(y_rf_test, rf_pred)
roc_auc = roc_auc_score(y_rf_test, rf_prob)

cm = confusion_matrix(y_rf_test, rf_pred)

tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MODEL 4 — RANDOM FOREST RESULTS")
print("=" * 60)

print(f"Evaluation set: {len(y_rf_test)}")
print(f"Legitimate: {(y_rf_test == 0).sum()}")
print(f"DGA: {(y_rf_test == 1).sum()}")

print()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"FPR      : {fpr:.4f}")
print(f"FNR      : {fnr:.4f}")

print("\nConfusion Matrix:")
print(cm)

# ------------------------------------------------------------
# Family-wise recall
# ------------------------------------------------------------

family_lookup = (
    unseen_test[["domain", "family"]]
    .drop_duplicates(subset=["domain"])
    .set_index("domain")["family"]
)

family_values = family_lookup.loc[
    unseen_test["domain"]
].values

print("\nFamily recall:")

for family in ["bamital", "goz", "matsnu"]:

    mask = (
        (family_values == family) &
        (y_rf_test.values == 1)
    )

    if mask.sum() > 0:
        family_recall = (rf_pred[mask] == 1).mean()
        print(f"  {family.capitalize():8s}: {family_recall:.4f}")

print("=" * 60)

# ------------------------------------------------------------
# Store result
# ------------------------------------------------------------

score_store["random_forest"] = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "roc_auc": roc_auc,
    "fpr": fpr,
    "fnr": fnr,
    "confusion_matrix": cm.tolist()
}

print("\nRandom Forest result saved to score_store.")

Preparing Random Forest features...
Training shape: (109316, 24)
Test shape: (10828, 24)

Training Random Forest...
Training complete.

MODEL 4 — RANDOM FOREST RESULTS
Evaluation set: 10828
Legitimate: 5414
DGA: 5414

Accuracy : 0.8688
Precision: 0.9574
Recall   : 0.7719
F1       : 0.8547
ROC-AUC  : 0.9120
FPR      : 0.0344
FNR      : 0.2281

Confusion Matrix:
[[5228  186]
 [1235 4179]]

Family recall:
  Bamital : 1.0000
  Goz     : 1.0000
  Matsnu  : 0.1633

Random Forest result saved to score_store.


In [65]:
# ============================================================
# MODEL 5 — CHAR TF-IDF 1–5 + STRUCTURAL FEATURES
# Targeted experiment for short/pronounceable DGA
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from scipy.sparse import hstack, csr_matrix
import numpy as np
import gc

print("=" * 65)
print("MODEL 5 — CHAR TF-IDF 1–5 + STRUCTURAL FEATURES")
print("=" * 65)

# ------------------------------------------------------------
# 1. Prepare text
# ------------------------------------------------------------

train_text = unseen_train["dga_string"].fillna("").astype(str)
test_text = unseen_test["dga_string"].fillna("").astype(str)

y5_train = unseen_train["label"].astype(int).values
y5_test = unseen_test["label"].astype(int).values

# ------------------------------------------------------------
# 2. Character TF-IDF
# ------------------------------------------------------------

print("\nBuilding character TF-IDF 1–5...")

tfidf5 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(1, 5),
    min_df=3,
    max_features=300000,
    sublinear_tf=True
)

X_char5_train = tfidf5.fit_transform(train_text)
X_char5_test = tfidf5.transform(test_text)

print("Character train shape:", X_char5_train.shape)
print("Character test shape :", X_char5_test.shape)

# ------------------------------------------------------------
# 3. Structural features
# ------------------------------------------------------------

print("\nAdding structural features...")

# Use the SAME domain-based alignment approach as RF
struct_lookup = (
    features_df[["domain"] + feature_columns]
    .drop_duplicates(subset=["domain"])
    .set_index("domain")
)

X_struct_train = struct_lookup.loc[
    unseen_train["domain"],
    feature_columns
].astype(np.float32)

X_struct_test = struct_lookup.loc[
    unseen_test["domain"],
    feature_columns
].astype(np.float32)

# Standardize structural features
scaler5 = StandardScaler()

X_struct_train_scaled = scaler5.fit_transform(X_struct_train)
X_struct_test_scaled = scaler5.transform(X_struct_test)

X_struct_train_sparse = csr_matrix(X_struct_train_scaled)
X_struct_test_sparse = csr_matrix(X_struct_test_scaled)

# ------------------------------------------------------------
# 4. Combine
# ------------------------------------------------------------

X5_train = hstack(
    [X_char5_train, X_struct_train_sparse],
    format="csr"
)

X5_test = hstack(
    [X_char5_test, X_struct_test_sparse],
    format="csr"
)

print("Combined train shape:", X5_train.shape)
print("Combined test shape :", X5_test.shape)

# ------------------------------------------------------------
# 5. Train SVM
# ------------------------------------------------------------

print("\nTraining Model 5...")

model5 = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

model5.fit(X5_train, y5_train)

print("Training complete.")

# ------------------------------------------------------------
# 6. Predictions
# ------------------------------------------------------------

pred5 = model5.predict(X5_test)
score5 = model5.decision_function(X5_test)

# ------------------------------------------------------------
# 7. Overall metrics
# ------------------------------------------------------------

accuracy5 = accuracy_score(y5_test, pred5)
precision5 = precision_score(y5_test, pred5)
recall5 = recall_score(y5_test, pred5)
f15 = f1_score(y5_test, pred5)
auc5 = roc_auc_score(y5_test, score5)

cm5 = confusion_matrix(y5_test, pred5)

tn5, fp5, fn5, tp5 = cm5.ravel()

fpr5 = fp5 / (fp5 + tn5)
fnr5 = fn5 / (fn5 + tp5)

print("\n" + "=" * 65)
print("MODEL 5 RESULTS")
print("=" * 65)

print(f"Evaluation set: {len(y5_test)}")
print(f"Legitimate: {(y5_test == 0).sum()}")
print(f"DGA: {(y5_test == 1).sum()}")

print()

print(f"Accuracy : {accuracy5:.4f}")
print(f"Precision: {precision5:.4f}")
print(f"Recall   : {recall5:.4f}")
print(f"F1       : {f15:.4f}")
print(f"ROC-AUC  : {auc5:.4f}")
print(f"FPR      : {fpr5:.4f}")
print(f"FNR      : {fnr5:.4f}")

print("\nConfusion Matrix:")
print(cm5)

# ------------------------------------------------------------
# 8. Family-wise recall
# ------------------------------------------------------------

family_lookup5 = (
    unseen_test[["domain", "family"]]
    .drop_duplicates("domain")
    .set_index("domain")["family"]
)

family_values5 = family_lookup5.loc[
    unseen_test["domain"]
].values

print("\nFamily recall:")

for family in ["bamital", "goz", "matsnu"]:

    mask = (
        (family_values5 == family) &
        (y5_test == 1)
    )

    if mask.sum() > 0:
        fam_recall = (pred5[mask] == 1).mean()
        print(f"  {family.capitalize():8s}: {fam_recall:.4f}")

# ------------------------------------------------------------
# 9. SHORT DGA ANALYSIS
# ------------------------------------------------------------

dga_mask5 = y5_test == 1
dga_lengths5 = unseen_test["dga_string"].str.len().values

print("\nShort DGA recall:")

for max_len in [8, 10, 12, 15]:

    mask = dga_mask5 & (dga_lengths5 <= max_len)

    if mask.sum() > 0:
        short_recall = (pred5[mask] == 1).mean()
        print(
            f"  Length <= {max_len:2d}: "
            f"{short_recall:.4f} "
            f"({mask.sum()} samples)"
        )

# ------------------------------------------------------------
# 10. Specific challenge domains
# ------------------------------------------------------------

challenge_domains5 = [
    "spendcroud.tk",
    "lordsleep.com",
    "javali.org",
    "gilma.mn",
    "xeogrhxquuubt.com"
]

print("\nChallenge domains:")

for domain in challenge_domains5:

    matches = np.where(
        unseen_test["domain"].values == domain
    )[0]

    if len(matches) == 0:
        print(f"  {domain:25s} -> not in unseen test")
        continue

    idx = matches[0]

    print(
        f"  {domain:25s} -> "
        f"prediction={pred5[idx]}, "
        f"score={score5[idx]:.4f}, "
        f"label={y5_test[idx]}"
    )

# ------------------------------------------------------------
# 11. Save result
# ------------------------------------------------------------

score_store["char_svm_1_5_structural"] = {
    "accuracy": accuracy5,
    "precision": precision5,
    "recall": recall5,
    "f1": f15,
    "roc_auc": auc5,
    "fpr": fpr5,
    "fnr": fnr5,
    "confusion_matrix": cm5.tolist()
}

print("\nModel 5 result saved to score_store.")

# ------------------------------------------------------------
# 12. RAM cleanup
# ------------------------------------------------------------

del (
    X_char5_train,
    X_char5_test,
    X_struct_train,
    X_struct_test,
    X_struct_train_scaled,
    X_struct_test_scaled,
    X_struct_train_sparse,
    X_struct_test_sparse,
    X5_train,
    X5_test
)

gc.collect()

print("RAM cleanup complete.")

MODEL 5 — CHAR TF-IDF 1–5 + STRUCTURAL FEATURES

Building character TF-IDF 1–5...
Character train shape: (109316, 168157)
Character test shape : (10828, 168157)

Adding structural features...
Combined train shape: (109316, 168181)
Combined test shape : (10828, 168181)

Training Model 5...
Training complete.

MODEL 5 RESULTS
Evaluation set: 10828
Legitimate: 5414
DGA: 5414

Accuracy : 0.9158
Precision: 0.9996
Recall   : 0.8319
F1       : 0.9081
ROC-AUC  : 0.9871
FPR      : 0.0004
FNR      : 0.1681

Confusion Matrix:
[[5412    2]
 [ 910 4504]]

Family recall:
  Bamital : 0.9949
  Goz     : 1.0000
  Matsnu  : 0.3869

Short DGA recall:
  Length <= 12: 0.1000 (10 samples)
  Length <= 15: 0.1290 (31 samples)

Challenge domains:
  spendcroud.tk             -> not in unseen test
  lordsleep.com             -> not in unseen test
  javali.org                -> not in unseen test
  gilma.mn                  -> not in unseen test
  xeogrhxquuubt.com         -> not in unseen test

Model 5 result sa

In [66]:
# ============================================================
# MODEL 5 — FAIR EVALUATION ON ORIGINAL 43,442 TEST SET
# ============================================================

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Reconstruct the ORIGINAL unseen-family test
# ------------------------------------------------------------

held_out_families = ["bamital", "goz", "matsnu"]

held_out_dga = final_dataset[
    (final_dataset["label"] == 1) &
    (final_dataset["family"].isin(held_out_families))
].copy()

legitimate = final_dataset[
    final_dataset["label"] == 0
].copy()

original_test_legit = legitimate.sample(
    n=len(held_out_dga),
    random_state=42
)

original_unseen_test = pd.concat(
    [held_out_dga, original_test_legit],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Original unseen-family test shape:",
      original_unseen_test.shape)

# ------------------------------------------------------------
# 2. Build Model 5 features
# ------------------------------------------------------------

test_text_original = (
    original_unseen_test["dga_string"]
    .fillna("")
    .astype(str)
)

y_original = (
    original_unseen_test["label"]
    .astype(int)
    .values
)

print("\nTransforming character features...")

X_char_original = tfidf5.transform(
    test_text_original
)

# Structural features
struct_original = (
    features_df[["domain"] + feature_columns]
    .drop_duplicates(subset=["domain"])
    .set_index("domain")
)

X_struct_original = struct_original.loc[
    original_unseen_test["domain"],
    feature_columns
].astype(np.float32)

X_struct_original_scaled = scaler5.transform(
    X_struct_original
)

X_struct_original_sparse = csr_matrix(
    X_struct_original_scaled
)

# Combine
X_original = hstack(
    [
        X_char_original,
        X_struct_original_sparse
    ],
    format="csr"
)

print("Original test feature shape:",
      X_original.shape)

# ------------------------------------------------------------
# 3. Predict using ALREADY TRAINED Model 5
# ------------------------------------------------------------

print("\nRunning Model 5 prediction...")

pred_original = model5.predict(X_original)
score_original = model5.decision_function(X_original)

# ------------------------------------------------------------
# 4. Metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_original, pred_original)
precision = precision_score(y_original, pred_original)
recall = recall_score(y_original, pred_original)
f1 = f1_score(y_original, pred_original)
roc_auc = roc_auc_score(y_original, score_original)

cm = confusion_matrix(
    y_original,
    pred_original
)

tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print("\n" + "=" * 65)
print("MODEL 5 — ORIGINAL 43,442 TEST SET")
print("=" * 65)

print(f"Evaluation set: {len(y_original)}")
print(f"Legitimate: {(y_original == 0).sum()}")
print(f"DGA: {(y_original == 1).sum()}")

print()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"FPR      : {fpr:.4f}")
print(f"FNR      : {fnr:.4f}")

print("\nConfusion Matrix:")
print(cm)

# ------------------------------------------------------------
# 5. Family recall
# ------------------------------------------------------------

family_lookup = (
    original_unseen_test[
        ["domain", "family"]
    ]
    .drop_duplicates("domain")
    .set_index("domain")["family"]
)

family_values = family_lookup.loc[
    original_unseen_test["domain"]
].values

print("\nFamily recall:")

for family in ["bamital", "goz", "matsnu"]:

    mask = (
        (family_values == family) &
        (y_original == 1)
    )

    if mask.sum() > 0:

        family_recall = (
            pred_original[mask] == 1
        ).mean()

        print(
            f"  {family.capitalize():8s}: "
            f"{family_recall:.4f}"
        )

# ------------------------------------------------------------
# 6. Short-DGA recall
# ------------------------------------------------------------

dga_mask = y_original == 1

lengths = (
    original_unseen_test[
        "dga_string"
    ].str.len().values
)

print("\nShort DGA recall:")

for max_len in [8, 10, 12, 15]:

    mask = (
        dga_mask &
        (lengths <= max_len)
    )

    if mask.sum() > 0:

        short_recall = (
            pred_original[mask] == 1
        ).mean()

        print(
            f"  Length <= {max_len:2d}: "
            f"{short_recall:.4f} "
            f"({mask.sum()} samples)"
        )
    else:
        print(
            f"  Length <= {max_len:2d}: "
            f"No samples"
        )

print("=" * 65)

Original unseen-family test shape: (10828, 7)

Transforming character features...
Original test feature shape: (10828, 168181)

Running Model 5 prediction...

MODEL 5 — ORIGINAL 43,442 TEST SET
Evaluation set: 10828
Legitimate: 5414
DGA: 5414

Accuracy : 0.9158
Precision: 0.9996
Recall   : 0.8319
F1       : 0.9081
ROC-AUC  : 0.9871
FPR      : 0.0004
FNR      : 0.1681

Confusion Matrix:
[[5412    2]
 [ 910 4504]]

Family recall:
  Bamital : 0.9949
  Goz     : 1.0000
  Matsnu  : 0.3869

Short DGA recall:
  Length <=  8: No samples
  Length <= 10: No samples
  Length <= 12: 0.1000 (10 samples)
  Length <= 15: 0.1290 (31 samples)


In [67]:
print("final_dataset:", final_dataset.shape)
print("unseen_train:", unseen_train.shape)
print("unseen_test:", unseen_test.shape)

print("\nFinal dataset labels:")
print(final_dataset["label"].value_counts())

print("\nFamilies:")
print(
    final_dataset[
        final_dataset["family"].isin(["bamital", "goz", "matsnu"])
    ]["family"].value_counts()
)

final_dataset: (114730, 7)
unseen_train: (109316, 7)
unseen_test: (10828, 7)

Final dataset labels:
label
1    74280
0    40450
Name: count, dtype: int64

Families:
family
goz        2967
matsnu     1476
bamital     971
Name: count, dtype: int64


In [68]:
# ============================================================
# RECOVER ORIGINAL FULL DATASET + 43,442 TEST SET
# ============================================================

import pandas as pd
import numpy as np

print("Loading original processed dataset...")

# Load the FULL dataset
full_dataset = pd.read_csv(
    "/content/final_dga_training_dataset.csv"
)

print("Full dataset:", full_dataset.shape)

# ------------------------------------------------------------
# Recreate the exact original unseen-family split
# ------------------------------------------------------------

held_out_families = ["bamital", "goz", "matsnu"]

held_out_dga_full = full_dataset[
    (full_dataset["label"] == 1) &
    (full_dataset["family"].isin(held_out_families))
].copy()

seen_dga_full = full_dataset[
    (full_dataset["label"] == 1) &
    (~full_dataset["family"].isin(held_out_families))
].copy()

legitimate_full = full_dataset[
    full_dataset["label"] == 0
].copy()

# Original training set
original_unseen_train = pd.concat(
    [legitimate_full, seen_dga_full],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Original balanced test set
original_test_legit = legitimate_full.sample(
    n=len(held_out_dga_full),
    random_state=42
)

original_unseen_test = pd.concat(
    [
        held_out_dga_full,
        original_test_legit
    ],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\n" + "=" * 60)
print("RECOVERED ORIGINAL SPLIT")
print("=" * 60)

print("Training:", original_unseen_train.shape)
print("Test    :", original_unseen_test.shape)

print("\nTest labels:")
print(original_unseen_test["label"].value_counts())

print("\nHeld-out DGA families:")
print(
    original_unseen_test[
        original_unseen_test["label"] == 1
    ]["family"].value_counts()
)

print("=" * 60)

Loading original processed dataset...
Full dataset: (114730, 7)

RECOVERED ORIGINAL SPLIT
Training: (109316, 7)
Test    : (10828, 7)

Test labels:
label
1    5414
0    5414
Name: count, dtype: int64

Held-out DGA families:
family
goz        2967
matsnu     1476
bamital     971
Name: count, dtype: int64


In [70]:
# ============================================================
# MODEL 5 — FAIR EVALUATION ON ORIGINAL 43,442 TEST
# FIXED: CALCULATE STRUCTURAL FEATURES DIRECTLY
# ============================================================

import pandas as pd
import numpy as np
import math
from collections import Counter
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ============================================================
# 1. LOAD ORIGINAL DATASET — READ ONLY
# ============================================================

print("=" * 70)
print("LOADING ORIGINAL DATASET")
print("=" * 70)

original_full = pd.read_csv(
    "/content/final_dga_training_dataset_ORIGINAL.csv"
)

print("Original dataset:", original_full.shape)

assert len(original_full) == 768246

# ============================================================
# 2. RECREATE ORIGINAL UNSEEN-FAMILY TEST
# ============================================================

held_out_families = ["bamital", "goz", "matsnu"]

held_out_dga = original_full[
    (original_full["label"] == 1) &
    (original_full["family"].isin(held_out_families))
].copy()

legitimate = original_full[
    original_full["label"] == 0
].copy()

test_legit = legitimate.sample(
    n=len(held_out_dga),
    random_state=42
)

original_test_43442 = pd.concat(
    [
        held_out_dga,
        test_legit
    ],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nOriginal test:", original_test_43442.shape)
print(
    original_test_43442["label"].value_counts()
)

print("\nFamily distribution:")
print(
    original_test_43442[
        original_test_43442["label"] == 1
    ]["family"].value_counts()
)

# ============================================================
# 3. EXACT 24-FEATURE EXTRACTOR
# ============================================================

VOWELS = set("aeiou")

def shannon_entropy_local(s):
    if not s:
        return 0.0

    counts = Counter(s)
    n = len(s)

    return -sum(
        (count / n) * math.log2(count / n)
        for count in counts.values()
    )


def max_consecutive_local(s, condition):
    best = 0
    current = 0

    for char in s:
        if condition(char):
            current += 1
            best = max(best, current)
        else:
            current = 0

    return best


def extract_features_for_original(domain, dga_string=None):

    domain = str(domain).lower().strip()

    if dga_string is None:
        dga_string = domain

    s = str(dga_string).lower().strip()

    s = "".join(
        c for c in s
        if c.isalnum() or c == "-"
    )

    length = len(s)

    labels = domain.split(".") if domain else []

    dot_count = domain.count(".")

    subdomain_count = max(
        len(labels) - 2,
        0
    )

    special_char_count = sum(
        not c.isalnum()
        for c in domain
    )

    special_char_ratio = (
        special_char_count / len(domain)
        if domain else 0
    )

    if length == 0:

        return {
            "domain_length": 0,
            "num_digits": 0,
            "digit_ratio": 0.0,
            "num_letters": 0,
            "num_vowels": 0,
            "num_consonants": 0,
            "vowel_ratio": 0.0,
            "consonant_ratio": 0.0,
            "num_special_chars": special_char_count,
            "special_char_ratio": special_char_ratio,
            "hyphen_count": domain.count("-"),
            "dot_count": dot_count,
            "subdomain_count": subdomain_count,
            "unique_chars": 0,
            "unique_char_ratio": 0.0,
            "shannon_entropy": 0.0,
            "max_consecutive_digits": 0,
            "max_consecutive_consonants": 0,
            "character_transitions": 0,
            "repeated_chars": 0,
            "repeated_ratio": 0.0,
            "max_consecutive_vowels": 0,
            "letter_digit_transitions": 0,
            "digit_letter_transitions": 0,
        }

    num_digits = sum(
        c.isdigit() for c in s
    )

    num_letters = sum(
        c.isalpha() for c in s
    )

    num_vowels = sum(
        c in VOWELS for c in s
    )

    num_consonants = sum(
        c.isalpha() and c not in VOWELS
        for c in s
    )

    unique_chars = len(set(s))

    repeated_chars = (
        length - unique_chars
    )

    digit_ratio = (
        num_digits / length
    )

    vowel_ratio = (
        num_vowels / length
    )

    consonant_ratio = (
        num_consonants / num_letters
        if num_letters else 0.0
    )

    unique_char_ratio = (
        unique_chars / length
    )

    repeated_ratio = (
        repeated_chars / length
    )

    max_consecutive_digits = max_consecutive_local(
        s,
        lambda c: c.isdigit()
    )

    max_consecutive_vowels = max_consecutive_local(
        s,
        lambda c:
            c.isalpha() and c in VOWELS
    )

    max_consecutive_consonants = max_consecutive_local(
        s,
        lambda c:
            c.isalpha() and c not in VOWELS
    )

    letter_digit_transitions = 0
    digit_letter_transitions = 0
    character_transitions = 0

    for i in range(1, len(s)):

        previous = s[i - 1]
        current = s[i]

        if (
            previous.isalpha()
            and current.isdigit()
        ):
            letter_digit_transitions += 1

        if (
            previous.isdigit()
            and current.isalpha()
        ):
            digit_letter_transitions += 1

        if previous != current:
            character_transitions += 1

    return {
        "domain_length": length,
        "num_digits": num_digits,
        "digit_ratio": digit_ratio,
        "num_letters": num_letters,
        "num_vowels": num_vowels,
        "num_consonants": num_consonants,
        "vowel_ratio": vowel_ratio,
        "consonant_ratio": consonant_ratio,
        "num_special_chars": special_char_count,
        "special_char_ratio": special_char_ratio,
        "hyphen_count": domain.count("-"),
        "dot_count": dot_count,
        "subdomain_count": subdomain_count,
        "unique_chars": unique_chars,
        "unique_char_ratio": unique_char_ratio,
        "shannon_entropy": shannon_entropy_local(s),
        "max_consecutive_digits": max_consecutive_digits,
        "max_consecutive_consonants":
            max_consecutive_consonants,
        "character_transitions":
            character_transitions,
        "repeated_chars":
            repeated_chars,
        "repeated_ratio":
            repeated_ratio,
        "max_consecutive_vowels":
            max_consecutive_vowels,
        "letter_digit_transitions":
            letter_digit_transitions,
        "digit_letter_transitions":
            digit_letter_transitions,
    }


# ============================================================
# 4. CALCULATE FEATURES DIRECTLY
# ============================================================

print("\nCalculating 24 structural features...")

feature_rows = []

for domain, dga_string in zip(
    original_test_43442["domain"],
    original_test_43442["dga_string"]
):

    feature_rows.append(
        extract_features_for_original(
            domain,
            dga_string
        )
    )

original_struct_df = pd.DataFrame(
    feature_rows
)

# Ensure exact feature order
original_struct_df = original_struct_df[
    feature_columns
]

print(
    "Structural feature shape:",
    original_struct_df.shape
)

# ============================================================
# 5. PREPARE CHARACTER FEATURES
# ============================================================

print("\nTransforming character TF-IDF...")

test_text_original = (
    original_test_43442["dga_string"]
    .fillna("")
    .astype(str)
)

X_char_original = tfidf5.transform(
    test_text_original
)

print(
    "Character feature shape:",
    X_char_original.shape
)

# ============================================================
# 6. SCALE STRUCTURAL FEATURES
# ============================================================

print("\nScaling structural features...")

X_struct_scaled = scaler5.transform(
    original_struct_df
)

X_struct_sparse = csr_matrix(
    X_struct_scaled
)

# ============================================================
# 7. COMBINE
# ============================================================

X_original_43442 = hstack(
    [
        X_char_original,
        X_struct_sparse
    ],
    format="csr"
)

print(
    "Combined feature shape:",
    X_original_43442.shape
)

# ============================================================
# 8. PREDICT WITH EXISTING MODEL 5
# ============================================================

print("\nRunning Model 5...")

y_original = (
    original_test_43442["label"]
    .astype(int)
    .values
)

pred_original = model5.predict(
    X_original_43442
)

score_original = model5.decision_function(
    X_original_43442
)

# ============================================================
# 9. METRICS
# ============================================================

accuracy = accuracy_score(
    y_original,
    pred_original
)

precision = precision_score(
    y_original,
    pred_original
)

recall = recall_score(
    y_original,
    pred_original
)

f1 = f1_score(
    y_original,
    pred_original
)

roc_auc = roc_auc_score(
    y_original,
    score_original
)

cm = confusion_matrix(
    y_original,
    pred_original
)

tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn)

fnr = fn / (fn + tp)

# ============================================================
# 10. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("MODEL 5 — TRUE ORIGINAL 43,442-SAMPLE TEST")
print("=" * 70)

print(
    f"Evaluation set: {len(y_original)}"
)

print(
    f"Legitimate: {(y_original == 0).sum()}"
)

print(
    f"DGA: {(y_original == 1).sum()}"
)

print()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"FPR      : {fpr:.4f}")
print(f"FNR      : {fnr:.4f}")

print("\nConfusion Matrix:")
print(cm)

# ============================================================
# 11. FAMILY RECALL
# ============================================================

print("\nFamily recall:")

for family in held_out_families:

    mask = (
        (original_test_43442["family"].values == family)
        &
        (y_original == 1)
    )

    family_recall = (
        pred_original[mask] == 1
    ).mean()

    print(
        f"  {family.capitalize():8s}: "
        f"{family_recall:.4f}"
    )

# ============================================================
# 12. SHORT-DGA RECALL
# ============================================================

lengths = (
    original_test_43442["dga_string"]
    .str.len()
    .values
)

print("\nShort DGA recall:")

for max_len in [8, 10, 12, 15]:

    mask = (
        (y_original == 1)
        &
        (lengths <= max_len)
    )

    if mask.sum() > 0:

        short_recall = (
            pred_original[mask] == 1
        ).mean()

        print(
            f"  Length <= {max_len:2d}: "
            f"{short_recall:.4f} "
            f"({mask.sum()} samples)"
        )

    else:

        print(
            f"  Length <= {max_len:2d}: "
            f"No samples"
        )

print("=" * 70)

print("\n✓ Evaluation complete.")
print("✓ ORIGINAL CSV was only READ.")
print("✓ Current CSV was not modified.")

LOADING ORIGINAL DATASET
Original dataset: (768246, 7)

Original test: (43442, 7)
label
0    21721
1    21721
Name: count, dtype: int64

Family distribution:
family
matsnu     13500
goz         6135
bamital     2086
Name: count, dtype: int64

Calculating 24 structural features...
Structural feature shape: (43442, 24)

Transforming character TF-IDF...
Character feature shape: (43442, 168157)

Scaling structural features...
Combined feature shape: (43442, 168181)

Running Model 5...

MODEL 5 — TRUE ORIGINAL 43,442-SAMPLE TEST
Evaluation set: 43442
Legitimate: 21721
DGA: 21721

Accuracy : 0.7837
Precision: 0.9285
Recall   : 0.6148
F1       : 0.7398
ROC-AUC  : 0.9213
FPR      : 0.0474
FNR      : 0.3852

Confusion Matrix:
[[20692  1029]
 [ 8367 13354]]

Family recall:
  Bamital : 0.9952
  Goz     : 1.0000
  Matsnu  : 0.3810

Short DGA recall:
  Length <=  8: No samples
  Length <= 10: No samples
  Length <= 12: 0.0980 (51 samples)
  Length <= 15: 0.0860 (279 samples)

✓ Evaluation complete.

In [71]:
# ============================================================
# FINAL THRESHOLD ANALYSIS — MODEL 5
# TRUE 43,442-SAMPLE UNSEEN-FAMILY TEST
# READ-ONLY — NO RETRAINING
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("MODEL 5 — THRESHOLD ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Use the scores already generated for the TRUE 43,442 test
# ------------------------------------------------------------

scores = score_original
y_true = y_original

print(f"Test samples: {len(y_true)}")
print(f"Legitimate : {(y_true == 0).sum()}")
print(f"DGA        : {(y_true == 1).sum()}")

# ------------------------------------------------------------
# Function to calculate metrics
# ------------------------------------------------------------

def evaluate_threshold(threshold):

    pred = (scores >= threshold).astype(int)

    cm = confusion_matrix(
        y_true,
        pred
    )

    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(
        y_true,
        pred
    )

    precision = precision_score(
        y_true,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        pred,
        zero_division=0
    )

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    return {
        "threshold": threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fpr": fpr,
        "fnr": fnr,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


# ------------------------------------------------------------
# Search a wide range of thresholds
# ------------------------------------------------------------

thresholds = np.linspace(
    scores.min(),
    scores.max(),
    1001
)

results = []

for threshold in thresholds:

    results.append(
        evaluate_threshold(threshold)
    )

threshold_df = pd.DataFrame(
    results
)

# ------------------------------------------------------------
# Find best recall under different FPR limits
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST RECALL AT DIFFERENT FPR LIMITS")
print("=" * 70)

for fpr_limit in [0.01, 0.02, 0.03, 0.04, 0.05]:

    eligible = threshold_df[
        threshold_df["fpr"] <= fpr_limit
    ]

    if len(eligible) == 0:

        print(
            f"\nFPR <= {fpr_limit*100:.0f}%: "
            "No threshold found"
        )

        continue

    best = eligible.sort_values(
        ["recall", "f1"],
        ascending=False
    ).iloc[0]

    print(
        f"\nFPR <= {fpr_limit*100:.0f}%"
    )

    print(
        f"  Threshold : {best['threshold']:.4f}"
    )

    print(
        f"  Accuracy  : {best['accuracy']:.4f}"
    )

    print(
        f"  Precision : {best['precision']:.4f}"
    )

    print(
        f"  Recall    : {best['recall']:.4f}"
    )

    print(
        f"  F1        : {best['f1']:.4f}"
    )

    print(
        f"  FPR       : {best['fpr']:.4f}"
    )

    print(
        f"  FNR       : {best['fnr']:.4f}"
    )

    print(
        f"  FP        : {int(best['fp'])}"
    )

    print(
        f"  FN        : {int(best['fn'])}"
    )


# ------------------------------------------------------------
# Show selected useful thresholds
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("THRESHOLD TRADE-OFF")
print("=" * 70)

# Find thresholds closest to target FPR values
target_fprs = [
    0.005,
    0.01,
    0.02,
    0.03,
    0.04,
    0.05,
    0.10
]

for target in target_fprs:

    idx = (
        threshold_df["fpr"] - target
    ).abs().idxmin()

    row = threshold_df.loc[idx]

    print(
        f"\nTarget FPR ≈ {target*100:.1f}%"
    )

    print(
        f"  Threshold : {row['threshold']:.4f}"
    )

    print(
        f"  Precision : {row['precision']:.4f}"
    )

    print(
        f"  Recall    : {row['recall']:.4f}"
    )

    print(
        f"  F1        : {row['f1']:.4f}"
    )

    print(
        f"  FPR       : {row['fpr']:.4f}"
    )

    print(
        f"  FNR       : {row['fnr']:.4f}"
    )


# ------------------------------------------------------------
# Compare default threshold 0
# ------------------------------------------------------------

default_result = evaluate_threshold(0)

print("\n" + "=" * 70)
print("DEFAULT MODEL 5 THRESHOLD = 0")
print("=" * 70)

print(
    f"Accuracy : {default_result['accuracy']:.4f}"
)

print(
    f"Precision: {default_result['precision']:.4f}"
)

print(
    f"Recall   : {default_result['recall']:.4f}"
)

print(
    f"F1       : {default_result['f1']:.4f}"
)

print(
    f"FPR      : {default_result['fpr']:.4f}"
)

print(
    f"FNR      : {default_result['fnr']:.4f}"
)

print("=" * 70)

print("\n✓ Threshold analysis complete.")
print("✓ No model retraining performed.")
print("✓ No CSV files modified.")

MODEL 5 — THRESHOLD ANALYSIS
Test samples: 43442
Legitimate : 21721
DGA        : 21721

BEST RECALL AT DIFFERENT FPR LIMITS

FPR <= 1%
  Threshold : 0.6008
  Accuracy  : 0.7069
  Precision : 0.9777
  Recall    : 0.4235
  F1        : 0.5910
  FPR       : 0.0097
  FNR       : 0.5765
  FP        : 210
  FN        : 12523

FPR <= 2%
  Threshold : 0.3359
  Accuracy  : 0.7334
  Precision : 0.9617
  Recall    : 0.4863
  F1        : 0.6459
  FPR       : 0.0194
  FNR       : 0.5137
  FP        : 421
  FN        : 11159

FPR <= 3%
  Threshold : 0.1704
  Accuracy  : 0.7559
  Precision : 0.9476
  Recall    : 0.5418
  F1        : 0.6894
  FPR       : 0.0300
  FNR       : 0.4582
  FP        : 651
  FN        : 9952

FPR <= 4%
  Threshold : 0.0601
  Accuracy  : 0.7739
  Precision : 0.9365
  Recall    : 0.5877
  F1        : 0.7222
  FPR       : 0.0399
  FNR       : 0.4123
  FP        : 866
  FN        : 8956

FPR <= 5%
  Threshold : -0.0062
  Accuracy  : 0.7847
  Precision : 0.9277
  Recall    : 0.617

In [73]:
# ================================================================
# LOAD FROZEN MODEL 1 ARTIFACTS
# NO RETRAINING
# ================================================================

import os
import joblib

MODEL_DIR = "/content/models"

model1_path = os.path.join(
    MODEL_DIR, "char_svm_model.joblib"
)

vectorizer_path = os.path.join(
    MODEL_DIR, "char_tfidf_vectorizer.joblib"
)

config_path = os.path.join(
    MODEL_DIR, "model_config.joblib"
)

print("=" * 70)
print("LOADING FROZEN MODEL 1")
print("=" * 70)

print("\nModel file:", model1_path)
print("Vectorizer:", vectorizer_path)
print("Config    :", config_path)

assert os.path.exists(model1_path), \
    f"Missing: {model1_path}"

assert os.path.exists(vectorizer_path), \
    f"Missing: {vectorizer_path}"

model1 = joblib.load(model1_path)
vectorizer = joblib.load(vectorizer_path)

if os.path.exists(config_path):
    model1_config = joblib.load(config_path)
else:
    model1_config = None

print("\n✓ Model 1 loaded")
print("✓ TF-IDF vectorizer loaded")

print("\nModel type:")
print(type(model1))

print("\nVectorizer:")
print(vectorizer)

if model1_config is not None:
    print("\nModel configuration:")
    print(model1_config)

print("\n✓ Frozen Model 1 remains unchanged.")

LOADING FROZEN MODEL 1

Model file: /content/models/char_svm_model.joblib
Vectorizer: /content/models/char_tfidf_vectorizer.joblib
Config    : /content/models/model_config.joblib

✓ Model 1 loaded
✓ TF-IDF vectorizer loaded

Model type:
<class 'sklearn.svm._classes.LinearSVC'>

Vectorizer:
TfidfVectorizer(analyzer='char', max_features=300000, min_df=3,
                ngram_range=(2, 5), sublinear_tf=True)

Model configuration:
{'feature_columns': ['domain_length', 'num_digits', 'digit_ratio', 'num_letters', 'num_vowels', 'num_consonants', 'vowel_ratio', 'consonant_ratio', 'num_special_chars', 'special_char_ratio', 'hyphen_count', 'dot_count', 'subdomain_count', 'unique_chars', 'unique_char_ratio', 'shannon_entropy', 'max_consecutive_digits', 'max_consecutive_consonants', 'character_transitions', 'repeated_chars', 'repeated_ratio', 'max_consecutive_vowels', 'letter_digit_transitions', 'digit_letter_transitions'], 'held_out_families': ['bamital', 'goz', 'matsnu'], 'tfidf_ngram_range': (

In [75]:
# ================================================================
# REBUILD EXACT ORIGINAL 43,442 TEST
# THEN ALIGN MODEL 1 + MODEL 5 SCORES
#
# NO RETRAINING
# NO CSV MODIFICATION
# ================================================================

import os
import gc
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

print("=" * 70)
print("REBUILDING EXACT ORIGINAL TEST SET")
print("=" * 70)

# ------------------------------------------------
# 1. LOAD READ-ONLY ORIGINAL DATASET
# ------------------------------------------------

original_path = "/content/final_dga_training_dataset_ORIGINAL.csv"

assert os.path.exists(original_path), (
    f"Original backup not found: {original_path}"
)

original_df = pd.read_csv(original_path)

print("\nOriginal dataset:")
print(original_df.shape)

print(
    original_df["label"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------
# 2. RECREATE EXACT ORIGINAL UNSEEN-FAMILY SPLIT
# ------------------------------------------------

held_out_families = [
    "bamital",
    "goz",
    "matsnu"
]

held_out_dga = original_df[
    (original_df["label"] == 1) &
    (original_df["family"].isin(held_out_families))
].copy()

seen_dga = original_df[
    (original_df["label"] == 1) &
    (~original_df["family"].isin(held_out_families))
].copy()

legitimate = original_df[
    original_df["label"] == 0
].copy()


unseen_train_original = (
    pd.concat(
        [legitimate, seen_dga],
        ignore_index=True
    )
    .sample(
        frac=1,
        random_state=42
    )
    .reset_index(drop=True)
)


test_unseen_dga_original = (
    held_out_dga.copy()
)


test_unseen_legit_original = (
    legitimate
    .sample(
        n=len(test_unseen_dga_original),
        random_state=42
    )
)


unseen_test_original = (
    pd.concat(
        [
            test_unseen_dga_original,
            test_unseen_legit_original
        ],
        ignore_index=True
    )
    .sample(
        frac=1,
        random_state=42
    )
    .reset_index(drop=True)
)


print("\nReconstructed split:")
print(
    "Train:",
    unseen_train_original.shape
)

print(
    "Test :",
    unseen_test_original.shape
)

print(
    "\nTest labels:"
)

print(
    unseen_test_original["label"]
    .value_counts()
    .sort_index()
)

print(
    "\nTest families:"
)

print(
    unseen_test_original[
        unseen_test_original["label"] == 1
    ]["family"].value_counts()
)


# ------------------------------------------------
# 3. MAKE SURE THIS IS THE EXPECTED 43,442 TEST
# ------------------------------------------------

assert len(unseen_test_original) == 43442

assert (
    (unseen_test_original["label"] == 0).sum()
    == 21721
)

assert (
    (unseen_test_original["label"] == 1).sum()
    == 21721
)

print(
    "\n✓ Exact 43,442-domain test reconstructed."
)


# ------------------------------------------------
# 4. MODEL 1 SCORES
# ------------------------------------------------

assert "model1" in globals()
assert "vectorizer" in globals()

domains = (
    unseen_test_original["domain"]
    .astype(str)
    .tolist()
)

y_true = (
    unseen_test_original["label"]
    .astype(int)
    .to_numpy()
)


print("\nCalculating Model 1 scores...")

X_model1_test = vectorizer.transform(
    domains
)

scores1_original = (
    model1.decision_function(
        X_model1_test
    )
)

del X_model1_test
gc.collect()

print(
    "✓ Model 1 scores:",
    len(scores1_original)
)


# ------------------------------------------------
# 5. MODEL 5 SCORES
# ------------------------------------------------
#
# IMPORTANT:
# Model 5 uses:
#   char TF-IDF 1-5
#   + 24 structural features
#
# We need to recreate those features for EXACTLY
# these 43,442 domains.
#
# This assumes the Model 5 vectorizer/model variables
# from your previous experiment are still in memory.
# ------------------------------------------------

print("\nChecking Model 5 variables...")

possible_model5 = [
    "model5",
    "vectorizer5",
    "tfidf5",
    "model5_vectorizer",
    "char_vectorizer5"
]

print(
    "Available Model 5 candidates:",
    [
        x for x in possible_model5
        if x in globals()
    ]
)

REBUILDING EXACT ORIGINAL TEST SET

Original dataset:
(768246, 7)
label
0    350748
1    417498
Name: count, dtype: int64

Reconstructed split:
Train: (746525, 7)
Test : (43442, 7)

Test labels:
label
0    21721
1    21721
Name: count, dtype: int64

Test families:
family
matsnu     13500
goz         6135
bamital     2086
Name: count, dtype: int64

✓ Exact 43,442-domain test reconstructed.

Calculating Model 1 scores...
✓ Model 1 scores: 43442

Checking Model 5 variables...
Available Model 5 candidates: ['model5', 'tfidf5']


In [76]:
# ================================================================
# MODEL 1 + MODEL 5 — FINAL ALIGNED ENSEMBLE TEST
#
# Exact original 43,442-domain unseen-family test
# NO RETRAINING
# NO CSV MODIFICATION
# ================================================================

import gc
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

print("=" * 70)
print("MODEL 1 + MODEL 5 — ALIGNED TEST")
print("=" * 70)


# ------------------------------------------------
# 1. CHECK MODEL 5 OBJECTS
# ------------------------------------------------

assert "model5" in globals()
assert "tfidf5" in globals()

print("✓ Model 5 loaded")
print("✓ Model 5 TF-IDF vectorizer loaded")


# ------------------------------------------------
# 2. EXACT TEST DOMAINS
# ------------------------------------------------

domains = (
    unseen_test_original["domain"]
    .astype(str)
    .tolist()
)

y_true = (
    unseen_test_original["label"]
    .astype(int)
    .to_numpy()
)

assert len(domains) == 43442
assert len(y_true) == 43442

print("\nTest domains:", len(domains))
print("Test labels :", len(y_true))


# ------------------------------------------------
# 3. MODEL 5 CHARACTER FEATURES
# ------------------------------------------------

print("\nGenerating Model 5 character TF-IDF features...")

X5_char = tfidf5.transform(domains)

print(
    "✓ Character matrix:",
    X5_char.shape
)


# ------------------------------------------------
# 4. STRUCTURAL FEATURES
# ------------------------------------------------
#
# Uses the exact extractor from the project.
# ------------------------------------------------

feature_columns = [
    "domain_length",
    "num_digits",
    "digit_ratio",
    "num_letters",
    "num_vowels",
    "num_consonants",
    "vowel_ratio",
    "consonant_ratio",
    "num_special_chars",
    "special_char_ratio",
    "hyphen_count",
    "dot_count",
    "subdomain_count",
    "unique_chars",
    "unique_char_ratio",
    "shannon_entropy",
    "max_consecutive_digits",
    "max_consecutive_consonants",
    "character_transitions",
    "repeated_chars",
    "repeated_ratio",
    "max_consecutive_vowels",
    "letter_digit_transitions",
    "digit_letter_transitions"
]

print(
    "\nGenerating 24 structural features..."
)

structural_rows = []

for domain in domains:

    structural_rows.append(
        extract_features(domain)
    )

X5_structural_df = pd.DataFrame(
    structural_rows
)[feature_columns]


print(
    "✓ Structural matrix:",
    X5_structural_df.shape
)


# ------------------------------------------------
# 5. STANDARDIZE STRUCTURAL FEATURES
# ------------------------------------------------
#
# IMPORTANT:
# Model 5 was trained with standardized structural
# features.
#
# We need the SAME scaler used during Model 5 training.
# ------------------------------------------------

print("\nChecking for Model 5 structural scaler...")

possible_scalers = [
    "scaler5",
    "structural_scaler",
    "scaler_structural",
    "standard_scaler5"
]

available_scalers = [
    x for x in possible_scalers
    if x in globals()
]

print(
    "Available scalers:",
    available_scalers
)

MODEL 1 + MODEL 5 — ALIGNED TEST
✓ Model 5 loaded
✓ Model 5 TF-IDF vectorizer loaded

Test domains: 43442
Test labels : 43442

Generating Model 5 character TF-IDF features...
✓ Character matrix: (43442, 168157)

Generating 24 structural features...
✓ Structural matrix: (43442, 24)

Checking for Model 5 structural scaler...
Available scalers: ['scaler5']


In [77]:
# ================================================================
# FINAL MODEL 1 + MODEL 5 ENSEMBLE
#
# Exact original 43,442-domain unseen-family test
# NO RETRAINING
# NO CSV MODIFICATION
# ================================================================

from scipy.sparse import hstack, csr_matrix
import numpy as np
import pandas as pd
import gc

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

print("=" * 70)
print("FINAL MODEL 1 + MODEL 5 ENSEMBLE")
print("=" * 70)


# ------------------------------------------------
# 1. STANDARDIZE MODEL 5 STRUCTURAL FEATURES
# ------------------------------------------------

print("\nStandardizing structural features...")

X5_structural = scaler5.transform(
    X5_structural_df
)

X5_structural = csr_matrix(
    X5_structural
)

print(
    "✓ Standardized structural matrix:",
    X5_structural.shape
)


# ------------------------------------------------
# 2. COMBINE MODEL 5 FEATURES
# ------------------------------------------------

print("\nCombining Model 5 features...")

X5_combined = hstack(
    [
        X5_char,
        X5_structural
    ],
    format="csr"
)

print(
    "✓ Model 5 combined matrix:",
    X5_combined.shape
)


# ------------------------------------------------
# 3. MODEL 5 SCORES
# ------------------------------------------------

print("\nCalculating Model 5 scores...")

scores5_original = (
    model5.decision_function(
        X5_combined
    )
)

print(
    "✓ Model 5 scores:",
    len(scores5_original)
)


# ------------------------------------------------
# 4. MODEL 1 SCORES
# ------------------------------------------------
#
# scores1_original was already calculated against
# the SAME 43,442 domains.
# ------------------------------------------------

print(
    "\nModel 1 scores:",
    len(scores1_original)
)

print(
    "Model 5 scores:",
    len(scores5_original)
)

print(
    "Labels:",
    len(y_true)
)


# ------------------------------------------------
# 5. FINAL ALIGNMENT CHECK
# ------------------------------------------------

assert len(scores1_original) == 43442
assert len(scores5_original) == 43442
assert len(y_true) == 43442

print("\n✓ ALL THREE ARRAYS PERFECTLY ALIGNED")


# ------------------------------------------------
# 6. NORMALIZE SVM SCORES
# ------------------------------------------------

def robust_standardize(x):

    x = np.asarray(x, dtype=float)

    median = np.median(x)
    mad = np.median(
        np.abs(x - median)
    )

    if mad == 0:

        std = np.std(x)

        if std == 0:
            return x - median

        return (x - median) / std

    return (
        x - median
    ) / (1.4826 * mad)


z1 = robust_standardize(
    scores1_original
)

z5 = robust_standardize(
    scores5_original
)


# ------------------------------------------------
# 7. EVALUATION FUNCTION
# ------------------------------------------------

def evaluate_model(
    name,
    scores=None,
    predictions=None
):

    if predictions is None:
        predictions = (
            scores >= 0
        ).astype(int)

    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            predictions
        ).ravel()
    )

    result = {
        "model": name,
        "accuracy":
            accuracy_score(
                y_true,
                predictions
            ),
        "precision":
            precision_score(
                y_true,
                predictions,
                zero_division=0
            ),
        "recall":
            recall_score(
                y_true,
                predictions,
                zero_division=0
            ),
        "f1":
            f1_score(
                y_true,
                predictions,
                zero_division=0
            ),
        "fpr":
            fp / (fp + tn),
        "fnr":
            fn / (fn + tp),
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "tp": tp
    }

    if scores is not None:
        result["roc_auc"] = (
            roc_auc_score(
                y_true,
                scores
            )
        )
    else:
        result["roc_auc"] = np.nan

    return result


# ------------------------------------------------
# 8. INDIVIDUAL MODELS
# ------------------------------------------------

results = []

pred1 = (
    scores1_original >= 0
).astype(int)

pred5 = (
    scores5_original >= 0
).astype(int)


results.append(
    evaluate_model(
        "Model 1",
        scores1_original
    )
)

results.append(
    evaluate_model(
        "Model 5",
        scores5_original
    )
)


# ------------------------------------------------
# 9. SCORE FUSION
# ------------------------------------------------

fusion_scores = {}

for w1, w5 in [
    (0.25, 0.75),
    (0.50, 0.50),
    (0.75, 0.25)
]:

    name = (
        f"Fusion {int(w1*100)}/"
        f"{int(w5*100)}"
    )

    combined = (
        w1 * z1 +
        w5 * z5
    )

    fusion_scores[name] = combined

    results.append(
        evaluate_model(
            name,
            combined
        )
    )


# ------------------------------------------------
# 10. OR RULE
# ------------------------------------------------

pred_or = (
    (pred1 == 1) |
    (pred5 == 1)
).astype(int)

results.append(
    evaluate_model(
        "OR rule",
        predictions=pred_or
    )
)


# ------------------------------------------------
# 11. AND RULE
# ------------------------------------------------

pred_and = (
    (pred1 == 1) &
    (pred5 == 1)
).astype(int)

results.append(
    evaluate_model(
        "AND rule",
        predictions=pred_and
    )
)


# ------------------------------------------------
# 12. LEADERBOARD
# ------------------------------------------------

leaderboard = pd.DataFrame(
    results
)

leaderboard = leaderboard[
    [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "fpr",
        "fnr",
        "fp",
        "fn"
    ]
]

leaderboard = (
    leaderboard
    .sort_values(
        "f1",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n")
print("=" * 70)
print("FINAL ENSEMBLE LEADERBOARD")
print("=" * 70)

display(
    leaderboard.style.format({
        "accuracy": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "roc_auc": "{:.4f}",
        "fpr": "{:.4f}",
        "fnr": "{:.4f}"
    })
)


# ------------------------------------------------
# 13. BEST FUSION
# ------------------------------------------------

fusion_rows = leaderboard[
    leaderboard["model"].str.startswith(
        "Fusion"
    )
]

best_fusion_name = (
    fusion_rows.iloc[0]["model"]
)

best_fusion_score = (
    fusion_scores[
        best_fusion_name
    ]
)

best_fusion_pred = (
    best_fusion_score >= 0
).astype(int)


print("\n")
print("=" * 70)
print("BEST FUSION")
print("=" * 70)

print(
    "\nBest fusion:",
    best_fusion_name
)


# ------------------------------------------------
# 14. FAMILY RECALL
# ------------------------------------------------

family_test = (
    unseen_test_original.copy()
)

family_test["prediction"] = (
    best_fusion_pred
)

print("\nFamily recall:")

for family in [
    "bamital",
    "goz",
    "matsnu"
]:

    subset = family_test[
        (family_test["family"] == family) &
        (family_test["label"] == 1)
    ]

    recall = (
        subset["prediction"].sum()
        / len(subset)
    )

    print(
        f"{family:10s}: "
        f"{recall:.4f} "
        f"({subset['prediction'].sum()}/"
        f"{len(subset)})"
    )


# ------------------------------------------------
# 15. MODEL AGREEMENT
# ------------------------------------------------

print("\n")
print("=" * 70)
print("MODEL AGREEMENT")
print("=" * 70)

both_dga = (
    (pred1 == 1) &
    (pred5 == 1)
).sum()

model1_only = (
    (pred1 == 1) &
    (pred5 == 0)
).sum()

model5_only = (
    (pred1 == 0) &
    (pred5 == 1)
).sum()

both_legit = (
    (pred1 == 0) &
    (pred5 == 0)
).sum()

print(
    "Both predict DGA     :",
    both_dga
)

print(
    "Model 1 only DGA     :",
    model1_only
)

print(
    "Model 5 only DGA     :",
    model5_only
)

print(
    "Both predict Legit   :",
    both_legit
)


# ------------------------------------------------
# 16. CLEANUP LARGE MATRICES
# ------------------------------------------------

del X5_char
del X5_structural
del X5_combined
del X5_structural_df

gc.collect()


print("\n")
print("=" * 70)
print("✓ ENSEMBLE EXPERIMENT COMPLETE")
print("=" * 70)
print("✓ Exact 43,442-domain test")
print("✓ Model 1 frozen")
print("✓ Model 5 frozen")
print("✓ No retraining")
print("✓ No CSV modification")

FINAL MODEL 1 + MODEL 5 ENSEMBLE

Standardizing structural features...
✓ Standardized structural matrix: (43442, 24)

Combining Model 5 features...
✓ Model 5 combined matrix: (43442, 168181)

Calculating Model 5 scores...
✓ Model 5 scores: 43442

Model 1 scores: 43442
Model 5 scores: 43442
Labels: 43442

✓ ALL THREE ARRAYS PERFECTLY ALIGNED


FINAL ENSEMBLE LEADERBOARD


,model,accuracy,precision,recall,f1,roc_auc,fpr,fnr,fp,fn
0,Fusion 75/25,0.8913,0.8898,0.8934,0.8916,0.9576,0.1107,0.1066,2404,2316
1,Model 1,0.8888,0.9330,0.8378,0.8829,0.9620,0.0601,0.1622,1306,3523
2,OR rule,0.8881,0.9287,0.8407,0.8825,nan,0.0645,0.1593,1401,3461
3,Fusion 50/50,0.8775,0.8760,0.8796,0.8778,0.9490,0.1245,0.1204,2704,2616
4,Fusion 25/75,0.8585,0.8579,0.8594,0.8587,0.9365,0.1423,0.1406,3091,3054
5,Model 5,0.7586,0.9457,0.5487,0.6945,0.9207,0.0315,0.4513,684,9802
6,AND rule,0.7594,0.9527,0.5459,0.6941,nan,0.0271,0.4541,589,9864




BEST FUSION

Best fusion: Fusion 75/25

Family recall:
bamital   : 1.0000 (2086/2086)
goz       : 1.0000 (6135/6135)
matsnu    : 0.8284 (11184/13500)


MODEL AGREEMENT
Both predict DGA     : 12446
Model 1 only DGA     : 7058
Model 5 only DGA     : 157
Both predict Legit   : 23781


✓ ENSEMBLE EXPERIMENT COMPLETE
✓ Exact 43,442-domain test
✓ Model 1 frozen
✓ Model 5 frozen
✓ No retraining
✓ No CSV modification


In [78]:
# ================================================================
# FINAL FUSION 75/25 THRESHOLD ANALYSIS
# NO RETRAINING
# NO CSV MODIFICATION
# ================================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

scores = np.asarray(
    fusion_scores["Fusion 75/25"]
)

y = np.asarray(
    y_true
).astype(int)


def evaluate_threshold(threshold):

    pred = (
        scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = (
        confusion_matrix(
            y,
            pred
        ).ravel()
    )

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(
            y, pred, zero_division=0
        ),
        "recall": recall_score(
            y, pred, zero_division=0
        ),
        "f1": f1_score(
            y, pred, zero_division=0
        ),
        "fpr": fp / (fp + tn),
        "fnr": fn / (fn + tp),
        "fp": fp,
        "fn": fn
    }


# ------------------------------------------------
# FIND BEST RECALL UNDER FPR LIMITS
# ------------------------------------------------

thresholds = np.linspace(
    scores.min(),
    scores.max(),
    2001
)

all_results = pd.DataFrame(
    [
        evaluate_threshold(t)
        for t in thresholds
    ]
)


print("=" * 70)
print("FUSION 75/25 — FINAL THRESHOLD ANALYSIS")
print("=" * 70)


for fpr_limit in [
    0.01,
    0.02,
    0.03,
    0.04,
    0.05,
    0.06,
    0.07,
    0.08,
    0.10
]:

    valid = all_results[
        all_results["fpr"] <= fpr_limit
    ]

    if len(valid) == 0:
        continue

    best = valid.loc[
        valid["recall"].idxmax()
    ]

    print(
        f"\nFPR <= {fpr_limit*100:.0f}%"
    )

    print(
        f"Threshold : {best['threshold']:.4f}"
    )

    print(
        f"Accuracy  : {best['accuracy']:.4f}"
    )

    print(
        f"Precision : {best['precision']:.4f}"
    )

    print(
        f"Recall    : {best['recall']:.4f}"
    )

    print(
        f"F1        : {best['f1']:.4f}"
    )

    print(
        f"FPR       : {best['fpr']:.4f}"
    )

    print(
        f"FNR       : {best['fnr']:.4f}"
    )

    print(
        f"FP        : {int(best['fp'])}"
    )

    print(
        f"FN        : {int(best['fn'])}"
    )


# ------------------------------------------------
# SPECIFIC OPERATING POINTS
# ------------------------------------------------

print("\n")
print("=" * 70)
print("FUSION THRESHOLD TRADE-OFF")
print("=" * 70)


for target_fpr in [
    0.03,
    0.04,
    0.05,
    0.06,
    0.07,
    0.08,
    0.10
]:

    idx = (
        all_results["fpr"] - target_fpr
    ).abs().idxmin()

    row = all_results.loc[idx]

    print(
        f"\nTarget FPR ≈ {target_fpr*100:.0f}%"
    )

    print(
        f"Threshold : {row['threshold']:.4f}"
    )

    print(
        f"Precision : {row['precision']:.4f}"
    )

    print(
        f"Recall    : {row['recall']:.4f}"
    )

    print(
        f"F1        : {row['f1']:.4f}"
    )

    print(
        f"FPR       : {row['fpr']:.4f}"
    )

    print(
        f"FNR       : {row['fnr']:.4f}"
    )


print("\n")
print("=" * 70)
print("DEFAULT FUSION THRESHOLD = 0")
print("=" * 70)

default = evaluate_threshold(0)

for key in [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "fpr",
    "fnr"
]:

    print(
        f"{key.capitalize():10s}: "
        f"{default[key]:.4f}"
    )

print("\n✓ No retraining performed.")
print("✓ No CSV files modified.")

FUSION 75/25 — FINAL THRESHOLD ANALYSIS

FPR <= 1%
Threshold : 0.7503
Accuracy  : 0.7431
Precision : 0.9803
Recall    : 0.4962
F1        : 0.6588
FPR       : 0.0100
FNR       : 0.5038
FP        : 217
FN        : 10944

FPR <= 2%
Threshold : 0.5263
Accuracy  : 0.7954
Precision : 0.9683
Recall    : 0.6107
F1        : 0.7490
FPR       : 0.0200
FNR       : 0.3893
FP        : 434
FN        : 8456

FPR <= 3%
Threshold : 0.4106
Accuracy  : 0.8250
Precision : 0.9581
Recall    : 0.6797
F1        : 0.7952
FPR       : 0.0297
FNR       : 0.3203
FP        : 646
FN        : 6957

FPR <= 4%
Threshold : 0.3166
Accuracy  : 0.8493
Precision : 0.9490
Recall    : 0.7383
F1        : 0.8305
FPR       : 0.0396
FNR       : 0.2617
FP        : 861
FN        : 5684

FPR <= 5%
Threshold : 0.2444
Accuracy  : 0.8646
Precision : 0.9399
Recall    : 0.7790
F1        : 0.8519
FPR       : 0.0498
FNR       : 0.2210
FP        : 1081
FN        : 4801

FPR <= 6%
Threshold : 0.1829
Accuracy  : 0.8765
Precision : 0.9316
Recal

In [79]:
# ============================================================
# 🔒 FREEZE FINAL MODEL 5 + ENSEMBLE CONFIGURATION
# Does NOT overwrite any existing dataset or Model 1 files.
# ============================================================

import os
import joblib
import json

# ------------------------------------------------------------
# 1. Make sure the models directory exists
# ------------------------------------------------------------
os.makedirs("/content/models", exist_ok=True)

# ------------------------------------------------------------
# 2. Save Model 5 components
#    These variables should already exist in the notebook:
#       model5
#       tfidf5
#       scaler5
# ------------------------------------------------------------

model5_path = "/content/models/model5_char_structural.joblib"
tfidf5_path = "/content/models/model5_tfidf_vectorizer.joblib"
scaler5_path = "/content/models/model5_scaler.joblib"

joblib.dump(model5, model5_path)
joblib.dump(tfidf5, tfidf5_path)
joblib.dump(scaler5, scaler5_path)

print("✅ Model 5 saved:")
print(model5_path)
print(tfidf5_path)
print(scaler5_path)


# ------------------------------------------------------------
# 3. Save FINAL ensemble configuration
#
# Model 1 = 75%
# Model 5 = 25%
#
# Score normalization = robust median/MAD
# Operating threshold = 0.1793
# ------------------------------------------------------------

ensemble_config = {
    "ensemble_name": "Fusion 75/25",
    "model_1_weight": 0.75,
    "model_5_weight": 0.25,

    "score_normalization": "robust_median_MAD",

    "threshold": 0.1793,

    "model_1": {
        "model": "char_svm_model.joblib",
        "vectorizer": "char_tfidf_vectorizer.joblib",
        "config": "model_config.joblib"
    },

    "model_5": {
        "model": "model5_char_structural.joblib",
        "vectorizer": "model5_tfidf_vectorizer.joblib",
        "scaler": "model5_scaler.joblib"
    },

    "feature_count_model_5": 24,

    "notes": (
        "Final candidate selected after unseen-family evaluation. "
        "Fusion uses 75% Model 1 and 25% Model 5 after robust "
        "median/MAD score standardization."
    )
}

ensemble_path = "/content/models/ensemble_config.joblib"
joblib.dump(ensemble_config, ensemble_path)

print("\n✅ Ensemble configuration saved:")
print(ensemble_path)


# ------------------------------------------------------------
# 4. Also save a human-readable JSON copy
# ------------------------------------------------------------

ensemble_json_path = "/content/models/ensemble_config.json"

with open(ensemble_json_path, "w") as f:
    json.dump(ensemble_config, f, indent=4)

print(ensemble_json_path)


# ------------------------------------------------------------
# 5. Verify everything exists
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("🔒 FROZEN MODEL CHECK")
print("=" * 60)

files_to_check = [
    "/content/models/char_svm_model.joblib",
    "/content/models/char_tfidf_vectorizer.joblib",
    "/content/models/model_config.joblib",

    "/content/models/model5_char_structural.joblib",
    "/content/models/model5_tfidf_vectorizer.joblib",
    "/content/models/model5_scaler.joblib",

    "/content/models/ensemble_config.joblib",
    "/content/models/ensemble_config.json",
]

for path in files_to_check:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ {os.path.basename(path):40s} {size_mb:.2f} MB")
    else:
        print(f"❌ MISSING: {path}")

print("\n🔒 Model freeze completed.")

✅ Model 5 saved:
/content/models/model5_char_structural.joblib
/content/models/model5_tfidf_vectorizer.joblib
/content/models/model5_scaler.joblib

✅ Ensemble configuration saved:
/content/models/ensemble_config.joblib
/content/models/ensemble_config.json

🔒 FROZEN MODEL CHECK
✅ char_svm_model.joblib                    1.13 MB
✅ char_tfidf_vectorizer.joblib             4.80 MB
✅ model_config.joblib                      0.00 MB
✅ model5_char_structural.joblib            1.28 MB
✅ model5_tfidf_vectorizer.joblib           5.44 MB
✅ model5_scaler.joblib                     0.00 MB
✅ ensemble_config.joblib                   0.00 MB
✅ ensemble_config.json                     0.00 MB

🔒 Model freeze completed.


In [81]:
# ============================================================
# FINAL FAMILY-WISE RECALL
#
# Reconstructs the EXACT original 43,442-row test set
# and loads the FROZEN models from disk.
#
# Does NOT modify any CSV or model files.
# ============================================================

import os
import math
import joblib
import numpy as np
import pandas as pd

from collections import Counter
from scipy.sparse import hstack

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

ORIGINAL_DATASET = "/content/final_dga_training_dataset_ORIGINAL.csv"
MODEL_DIR = "/content/models"

MODEL1_PATH = os.path.join(MODEL_DIR, "char_svm_model.joblib")
VECT1_PATH = os.path.join(MODEL_DIR, "char_tfidf_vectorizer.joblib")

MODEL5_PATH = os.path.join(MODEL_DIR, "model5_char_structural.joblib")
VECT5_PATH = os.path.join(MODEL_DIR, "model5_tfidf_vectorizer.joblib")
SCALER5_PATH = os.path.join(MODEL_DIR, "model5_scaler.joblib")

FINAL_THRESHOLD = 0.1793


# ------------------------------------------------------------
# 2. Load ORIGINAL 768,246-row dataset
# ------------------------------------------------------------

print("Loading original dataset...")

final_dataset = pd.read_csv(ORIGINAL_DATASET)

print("Dataset shape:", final_dataset.shape)

assert len(final_dataset) == 768246, \
    "This is not the expected original 768,246-row dataset."

print("✅ Original dataset confirmed.")


# ------------------------------------------------------------
# 3. Reconstruct EXACT unseen-family test set
#
# Held-out families:
#   Bamital
#   GOZ
#   Matsnu
#
# Same construction used in the previous experiment.
# ------------------------------------------------------------

held_out_families = ["bamital", "goz", "matsnu"]

held_out_dga = final_dataset[
    (final_dataset["label"] == 1) &
    (final_dataset["family"].str.lower().isin(held_out_families))
].copy()

seen_dga = final_dataset[
    (final_dataset["label"] == 1) &
    (~final_dataset["family"].str.lower().isin(held_out_families))
].copy()

legitimate = final_dataset[
    final_dataset["label"] == 0
].copy()


unseen_train = (
    pd.concat([legitimate, seen_dga], ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

test_unseen_dga = held_out_dga.copy()

test_unseen_legit = legitimate.sample(
    n=len(test_unseen_dga),
    random_state=42
)

unseen_test = (
    pd.concat(
        [test_unseen_dga, test_unseen_legit],
        ignore_index=True
    )
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("\nExact test set:")
print("Shape:", unseen_test.shape)
print("Legit:", (unseen_test["label"] == 0).sum())
print("DGA:", (unseen_test["label"] == 1).sum())

assert unseen_test.shape[0] == 43442
assert (unseen_test["label"] == 1).sum() == 21721
assert (unseen_test["label"] == 0).sum() == 21721

print("✅ Exact 43,442-row test reconstructed.")


# ------------------------------------------------------------
# 4. Load frozen Model 1
# ------------------------------------------------------------

print("\nLoading Model 1...")

model1 = joblib.load(MODEL1_PATH)
vectorizer1 = joblib.load(VECT1_PATH)

print("✅ Model 1 loaded.")


# ------------------------------------------------------------
# 5. Score Model 1
# ------------------------------------------------------------

X_test_char1 = vectorizer1.transform(
    unseen_test["domain"].astype(str)
)

model1_scores = model1.decision_function(X_test_char1)

print("Model 1 scores:", len(model1_scores))


# ------------------------------------------------------------
# 6. Exact 24 structural features for Model 5
# ------------------------------------------------------------

VOWELS = set("aeiou")

def shannon_entropy(s):
    if not s:
        return 0.0

    counts = Counter(s)
    n = len(s)

    return -sum(
        (count / n) * math.log2(count / n)
        for count in counts.values()
    )


def max_consecutive(s, condition):
    best = 0
    current = 0

    for char in s:
        if condition(char):
            current += 1
            best = max(best, current)
        else:
            current = 0

    return best


def extract_features_exact(domain, dga_string):

    domain = str(domain).lower().strip()
    s = str(dga_string).lower().strip()

    s = "".join(
        c for c in s
        if c.isalnum() or c == "-"
    )

    length = len(s)

    labels = domain.split(".") if domain else []

    dot_count = domain.count(".")

    subdomain_count = max(
        len(labels) - 2,
        0
    )

    special_char_count = sum(
        not c.isalnum()
        for c in domain
    )

    special_char_ratio = (
        special_char_count / len(domain)
        if domain else 0
    )

    if length == 0:
        return {
            "domain_length": 0,
            "num_digits": 0,
            "digit_ratio": 0.0,
            "num_letters": 0,
            "num_vowels": 0,
            "num_consonants": 0,
            "vowel_ratio": 0.0,
            "consonant_ratio": 0.0,
            "num_special_chars": special_char_count,
            "special_char_ratio": special_char_ratio,
            "hyphen_count": domain.count("-"),
            "dot_count": dot_count,
            "subdomain_count": subdomain_count,
            "unique_chars": 0,
            "unique_char_ratio": 0.0,
            "shannon_entropy": 0.0,
            "max_consecutive_digits": 0,
            "max_consecutive_consonants": 0,
            "character_transitions": 0,
            "repeated_chars": 0,
            "repeated_ratio": 0.0,
            "max_consecutive_vowels": 0,
            "letter_digit_transitions": 0,
            "digit_letter_transitions": 0,
        }

    num_digits = sum(c.isdigit() for c in s)

    num_letters = sum(c.isalpha() for c in s)

    num_vowels = sum(
        c in VOWELS
        for c in s
    )

    num_consonants = sum(
        c.isalpha() and c not in VOWELS
        for c in s
    )

    unique_chars = len(set(s))

    repeated_chars = length - unique_chars

    digit_ratio = num_digits / length

    vowel_ratio = num_vowels / length

    consonant_ratio = (
        num_consonants / num_letters
        if num_letters else 0.0
    )

    unique_char_ratio = unique_chars / length

    repeated_ratio = repeated_chars / length

    max_consecutive_digits = max_consecutive(
        s,
        lambda c: c.isdigit()
    )

    max_consecutive_vowels = max_consecutive(
        s,
        lambda c: c.isalpha() and c in VOWELS
    )

    max_consecutive_consonants = max_consecutive(
        s,
        lambda c: c.isalpha() and c not in VOWELS
    )

    letter_digit_transitions = 0
    digit_letter_transitions = 0
    character_transitions = 0

    for i in range(1, len(s)):

        previous = s[i - 1]
        current = s[i]

        if previous.isalpha() and current.isdigit():
            letter_digit_transitions += 1

        if previous.isdigit() and current.isalpha():
            digit_letter_transitions += 1

        if previous != current:
            character_transitions += 1

    return {
        "domain_length": length,
        "num_digits": num_digits,
        "digit_ratio": digit_ratio,
        "num_letters": num_letters,
        "num_vowels": num_vowels,
        "num_consonants": num_consonants,
        "vowel_ratio": vowel_ratio,
        "consonant_ratio": consonant_ratio,
        "num_special_chars": special_char_count,
        "special_char_ratio": special_char_ratio,
        "hyphen_count": s.count("-"),
        "dot_count": dot_count,
        "subdomain_count": subdomain_count,
        "unique_chars": unique_chars,
        "unique_char_ratio": unique_char_ratio,
        "shannon_entropy": shannon_entropy(s),
        "max_consecutive_digits": max_consecutive_digits,
        "max_consecutive_consonants": max_consecutive_consonants,
        "character_transitions": character_transitions,
        "repeated_chars": repeated_chars,
        "repeated_ratio": repeated_ratio,
        "max_consecutive_vowels": max_consecutive_vowels,
        "letter_digit_transitions": letter_digit_transitions,
        "digit_letter_transitions": digit_letter_transitions,
    }


feature_columns = [
    "domain_length",
    "num_digits",
    "digit_ratio",
    "num_letters",
    "num_vowels",
    "num_consonants",
    "vowel_ratio",
    "consonant_ratio",
    "num_special_chars",
    "special_char_ratio",
    "hyphen_count",
    "dot_count",
    "subdomain_count",
    "unique_chars",
    "unique_char_ratio",
    "shannon_entropy",
    "max_consecutive_digits",
    "max_consecutive_consonants",
    "character_transitions",
    "repeated_chars",
    "repeated_ratio",
    "max_consecutive_vowels",
    "letter_digit_transitions",
    "digit_letter_transitions",
]


print("\nExtracting Model 5 structural features...")

structural_rows = [
    extract_features_exact(
        domain,
        dga_string
    )
    for domain, dga_string in zip(
        unseen_test["domain"],
        unseen_test["dga_string"]
    )
]

X_structural = pd.DataFrame(
    structural_rows
)[feature_columns].astype(np.float32)

print("Structural matrix:", X_structural.shape)


# ------------------------------------------------------------
# 7. Load frozen Model 5
# ------------------------------------------------------------

print("\nLoading Model 5...")

model5 = joblib.load(MODEL5_PATH)
vectorizer5 = joblib.load(VECT5_PATH)
scaler5 = joblib.load(SCALER5_PATH)

print("✅ Model 5 loaded.")


# ------------------------------------------------------------
# 8. Score Model 5
# ------------------------------------------------------------

X_char5 = vectorizer5.transform(
    unseen_test["domain"].astype(str)
)

X_structural_scaled = scaler5.transform(
    X_structural
)

X_test_model5 = hstack(
    [X_char5, X_structural_scaled],
    format="csr"
)

print("Model 5 matrix:", X_test_model5.shape)

model5_scores = model5.decision_function(
    X_test_model5
)

print("Model 5 scores:", len(model5_scores))


# ------------------------------------------------------------
# 9. Robust score normalization
# ------------------------------------------------------------

def robust_standardize(x):

    x = np.asarray(x, dtype=float)

    median = np.median(x)

    mad = np.median(
        np.abs(x - median)
    )

    if mad == 0:

        std = np.std(x)

        if std == 0:
            return x - median

        return (x - median) / std

    return (
        (x - median)
        / (1.4826 * mad)
    )


model1_robust = robust_standardize(
    model1_scores
)

model5_robust = robust_standardize(
    model5_scores
)


# ------------------------------------------------------------
# 10. FINAL Fusion 75/25
# ------------------------------------------------------------

fusion_75_25 = (
    0.75 * model1_robust
    +
    0.25 * model5_robust
)

assert len(fusion_75_25) == len(unseen_test)

final_predictions = (
    fusion_75_25 >= FINAL_THRESHOLD
).astype(int)

print("\n✅ Fusion scores aligned:")
print("Scores:", len(fusion_75_25))
print("Test:", len(unseen_test))


# ------------------------------------------------------------
# 11. Family-wise recall
# ------------------------------------------------------------

results = []

for family in ["bamital", "goz", "matsnu"]:

    family_mask = (
        unseen_test["label"].eq(1)
        &
        unseen_test["family"]
        .str.lower()
        .eq(family)
    )

    total = int(family_mask.sum())

    detected = int(
        final_predictions[family_mask.values].sum()
    )

    missed = total - detected

    recall = (
        detected / total
        if total > 0
        else 0
    )

    results.append({
        "Family": family.upper(),
        "Total DGA": total,
        "Detected": detected,
        "Missed": missed,
        "Recall": recall * 100
    })


family_results = pd.DataFrame(results)


# ------------------------------------------------------------
# 12. Display final result
# ------------------------------------------------------------

print("\n")
print("=" * 75)
print("FINAL FAMILY-WISE RECALL")
print("Fusion 75/25 | Threshold = 0.1793")
print("=" * 75)

display(
    family_results.style.format({
        "Recall": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 13. Overall DGA recall
# ------------------------------------------------------------

dga_mask = unseen_test["label"].eq(1)

overall_total = int(dga_mask.sum())

overall_detected = int(
    final_predictions[dga_mask.values].sum()
)

overall_missed = (
    overall_total - overall_detected
)

overall_recall = (
    overall_detected / overall_total * 100
)

print("\nOverall DGA:")
print(f"Total:    {overall_total:,}")
print(f"Detected: {overall_detected:,}")
print(f"Missed:   {overall_missed:,}")
print(f"Recall:   {overall_recall:.2f}%")

print("\n🔒 No files were modified.")

Loading original dataset...
Dataset shape: (768246, 7)
✅ Original dataset confirmed.

Exact test set:
Shape: (43442, 7)
Legit: 21721
DGA: 21721
✅ Exact 43,442-row test reconstructed.

Loading Model 1...
✅ Model 1 loaded.
Model 1 scores: 43442

Extracting Model 5 structural features...
Structural matrix: (43442, 24)

Loading Model 5...
✅ Model 5 loaded.
Model 5 matrix: (43442, 168181)
Model 5 scores: 43442

✅ Fusion scores aligned:
Scores: 43442
Test: 43442


FINAL FAMILY-WISE RECALL
Fusion 75/25 | Threshold = 0.1793


,Family,Total DGA,Detected,Missed,Recall
0,BAMITAL,2086,2085,1,99.95%
1,GOZ,6135,6135,0,100.00%
2,MATSNU,13500,9480,4020,70.22%



Overall DGA:
Total:    21,721
Detected: 17,700
Missed:   4,021
Recall:   81.49%

🔒 No files were modified.


In [82]:
# ============================================================
# UPDATE FINAL EXPERIMENT RESULTS
# ============================================================

from pathlib import Path

results_text = r"""
============================================================
CNLA PROJECT — DGA DETECTION EXPERIMENT RESULTS
============================================================

Project:
AI-Based Detection of Algorithmically Generated Malicious
Domains (DGA Detection) in DNS Traffic

============================================================
1. DATASET
============================================================

Final original training dataset:
- Rows: 768,246
- Columns: 7
- DGA: 417,498
- Legitimate: 350,748

DGA families represented:
- bamital
- cryptolocker
- gameoverdga
- goz
- matsnu
- necurs
- newgoz
- nivdort
- Additional families from DGA_domains_dataset

Important:
The original dataset is stored as:
final_dga_training_dataset_ORIGINAL.csv

This file is READ-ONLY and must not be modified.

The current few-shot subset is stored separately as:
final_dga_training_dataset.csv

This file is also READ-ONLY.

============================================================
2. FEATURE EXTRACTION
============================================================

24 structural/lexical features were engineered:

1. domain_length
2. num_digits
3. digit_ratio
4. num_letters
5. num_vowels
6. num_consonants
7. vowel_ratio
8. consonant_ratio
9. num_special_chars
10. special_char_ratio
11. hyphen_count
12. dot_count
13. subdomain_count
14. unique_chars
15. unique_char_ratio
16. shannon_entropy
17. max_consecutive_digits
18. max_consecutive_consonants
19. character_transitions
20. repeated_chars
21. repeated_ratio
22. max_consecutive_vowels
23. letter_digit_transitions
24. digit_letter_transitions

DGA string extraction uses public-suffix-aware domain parsing
to correctly handle domains such as:
- example.co.uk
- example.co.nz
- example.co.id

============================================================
3. BASELINE — CHAR SVM 2–5
============================================================

Configuration:
- Character TF-IDF
- n-gram range: 2–5
- min_df: 3
- max_features: 300,000
- sublinear_tf: True
- LinearSVC
- C = 1.0
- class_weight = balanced
- random_state = 42

------------------------------------------------------------
3A. RANDOM SPLIT
------------------------------------------------------------

Train/test:
- 80/20 stratified random split
- Test size: 153,650

Results:
- Accuracy: 95.49%
- Precision: 96.18%
- Recall: 95.49%
- F1: 95.83%
- ROC-AUC: 98.97%
- FPR: 4.51%
- FNR: 4.51%

Confusion matrix:
TN = 66,987
FP = 3,163
FN = 3,768
TP = 79,732

Interpretation:
The model performs strongly when train and test data have
similar family/distribution characteristics.

------------------------------------------------------------
3B. UNSEEN-FAMILY / ZERO-SHOT EVALUATION
------------------------------------------------------------

Held-out DGA families:
- Bamital
- GOZ
- Matsnu

Exact test set:
- Total: 43,442
- Legitimate: 21,721
- DGA: 21,721

The model was trained without the held-out DGA families.

Results:
- Accuracy: 75.09%
- Precision: 97.65%
- Recall: 51.41%
- F1: 67.36%
- ROC-AUC: 91.21%
- FPR: 1.24%
- FNR: 48.59%

Confusion matrix:
TN = 21,452
FP = 269
FN = 10,554
TP = 11,167

Interpretation:
The baseline has very low false-positive rate but misses a
large number of unseen-family DGA domains. This exposed the
main generalization problem of the project.

============================================================
4. ALTERNATIVE MODELS TESTED
============================================================

------------------------------------------------------------
4A. CHAR SVM 2–6
------------------------------------------------------------

Unseen-family evaluation:
- Accuracy: 75.45%
- Precision: 97.06%
- Recall: 52.48%
- F1: 68.12%
- ROC-AUC: 91.24%
- FPR: 1.59%
- FNR: 47.52%

Family recall:
- Bamital: 48.90%
- GOZ: 100.00%
- Matsnu: 31.44%

Decision:
REJECTED.
Only marginal improvement over Char SVM 2–5 and poor Matsnu
generalization.

------------------------------------------------------------
4B. CHAR SVM 3–5
------------------------------------------------------------

Unseen-family evaluation:
- Accuracy: 71.43%
- Precision: 96.50%
- Recall: 44.47%
- F1: 60.88%
- ROC-AUC: 86.51%
- FPR: 1.61%
- FNR: 55.53%

Family recall:
- Bamital: 51.77%
- GOZ: 99.97%
- Matsnu: 18.12%

Decision:
REJECTED.

------------------------------------------------------------
4C. RANDOM FOREST
------------------------------------------------------------

Unseen-family evaluation:
- Accuracy: 68.64%
- Precision: 85.69%
- Recall: 44.76%
- F1: 58.80%
- ROC-AUC: 60.72%
- FPR: 7.47%
- FNR: 55.24%

Family recall:
- Bamital: 100.00%
- GOZ: 100.00%
- Matsnu: 11.12%

Decision:
REJECTED.

------------------------------------------------------------
4D. CHAR + BIGRAM/TRIGRAM LANGUAGE MODEL
------------------------------------------------------------

Random-split evaluation:
- Accuracy: 95.21%
- Precision: 95.75%
- Recall: 95.51%
- F1: 95.63%
- FPR: 5.16%
- FNR: 4.49%

Decision:
REJECTED as final model.
Did not provide sufficient improvement for the target
unseen-family/generalization problem.

------------------------------------------------------------
4E. WORD-LEVEL TF-IDF + LOGISTIC REGRESSION
------------------------------------------------------------

Unseen-family evaluation:
- Accuracy: 59.16%
- Precision: 55.12%
- Recall: 98.55%
- F1: 70.70%
- ROC-AUC: 59.26%
- FPR: 80.23%
- FNR: 1.45%

Confusion matrix:
TN = 4,295
FP = 17,426
FN = 315
TP = 21,406

Family recall:
- Bamital: 100.00%
- GOZ: 100.00%
- Matsnu: 97.67%

Decision:
REJECTED.
Excellent DGA recall but unacceptable false-positive rate.

============================================================
5. MODEL 5 — CHAR TF-IDF + STRUCTURAL FEATURES
============================================================

Configuration:
- Character TF-IDF
- n-gram range: 1–5
- min_df: 3
- max_features: 300,000
- sublinear_tf: True
- 24 structural features
- Standardized structural features
- LinearSVC
- C = 1.0
- class_weight = balanced
- random_state = 42

Model 5 was trained using the few-shot/adapted training
configuration containing limited Matsnu exposure.

Evaluation on the exact original 43,442-row family-held-out
test set:

- Accuracy: 78.37%
- Precision: 92.85%
- Recall: 61.48%
- F1: 73.98%
- ROC-AUC: 92.13%
- FPR: 4.74%
- FNR: 38.52%

Confusion matrix:
TN = 20,692
FP = 1,029
FN = 8,367
TP = 13,354

Family recall:
- Bamital: 99.52%
- GOZ: 100.00%
- Matsnu: 38.10%

Interpretation:
Adding structural features improved DGA recall compared with
the zero-shot Char SVM baseline, but Matsnu remained difficult.

Decision:
RETAINED as a component of the final ensemble.

============================================================
6. FINAL ENSEMBLE — FUSION 75/25
============================================================

Final ensemble:
- Model 1 weight: 75%
- Model 5 weight: 25%

Before fusion, each model's decision scores are robustly
standardized using median/MAD normalization.

Operating threshold:
0.1793

The threshold was selected to target approximately 6% FPR.
For rigorous future evaluation, threshold selection should
ideally be performed using a separate validation set rather
than the final test set.

Exact test set:
- Total: 43,442
- Legitimate: 21,721
- DGA: 21,721

------------------------------------------------------------
FINAL RESULTS
------------------------------------------------------------

- Accuracy: 87.65%
- Precision: 93.10%
- Recall: 81.49%
- F1: 86.91%
- ROC-AUC: 95.76%
- FPR: 6.04%
- FNR: 18.51%

Confusion matrix at threshold 0.1793:
- False positives: 1,311
- False negatives: 4,021

------------------------------------------------------------
FAMILY-WISE RECALL
------------------------------------------------------------

Bamital:
- Total: 2,086
- Detected: 2,085
- Missed: 1
- Recall: 99.95%

GOZ:
- Total: 6,135
- Detected: 6,135
- Missed: 0
- Recall: 100.00%

Matsnu:
- Total: 13,500
- Detected: 9,480
- Missed: 4,020
- Recall: 70.22%

Overall DGA:
- Total: 21,721
- Detected: 17,700
- Missed: 4,021
- Recall: 81.49%

============================================================
7. FINAL MODEL COMPARISON
============================================================

Model                         Accuracy  Precision  Recall   F1
----------------------------------------------------------------
Char SVM 2–5 (Random Split)     95.49%    96.18%   95.49%  95.83%
Char SVM 2–5 (Zero-shot)        75.09%    97.65%   51.41%  67.36%
Char SVM 2–6 (Zero-shot)        75.45%    97.06%   52.48%  68.12%
Char SVM 3–5 (Zero-shot)        71.43%    96.50%   44.47%  60.88%
Random Forest (Zero-shot)       68.64%    85.69%   44.76%  58.80%
Word TF-IDF + LR (Zero-shot)    59.16%    55.12%   98.55%  70.70%
Model 5                         78.37%    92.85%   61.48%  73.98%
Fusion 75/25                    87.65%    93.10%   81.49%  86.91%

FINAL MODEL:
Fusion 75/25

============================================================
8. WHY FUSION 75/25 WAS SELECTED
============================================================

The project prioritizes reducing false negatives while keeping
false positives at a practical level.

Compared with the zero-shot Char SVM 2–5 baseline:

Recall:
51.41% -> 81.49%

False-negative rate:
48.59% -> 18.51%

Precision:
97.65% -> 93.10%

False-positive rate:
1.24% -> 6.04%

The ensemble therefore substantially improves detection of
DGA domains at the cost of a moderate increase in false
positives.

============================================================
9. CURRENT PRODUCTION MODEL FILES
============================================================

models/
- char_svm_model.joblib
- char_tfidf_vectorizer.joblib
- model_config.joblib
- model5_char_structural.joblib
- model5_tfidf_vectorizer.joblib
- model5_scaler.joblib
- ensemble_config.joblib
- ensemble_config.json

These files are considered FROZEN.

Do not overwrite them without creating a new model version.

============================================================
10. DATA / MODEL FREEZE STATUS
============================================================

READ-ONLY DATASETS:
- final_dga_training_dataset_ORIGINAL.csv
- final_dga_training_dataset.csv

FROZEN MODELS:
- Model 1
- Model 5
- Fusion 75/25 ensemble configuration

FINAL OPERATING THRESHOLD:
0.1793

NEXT PROJECT PHASE:
Backend/API integration and single-domain prediction pipeline.

============================================================
11. IMPORTANT PROJECT LIMITATION
============================================================

This system performs lexical/structural DGA detection.

A DGA prediction does NOT by itself prove that a domain is
malicious.

The classifier does not directly observe:
- DNS query behaviour
- IP reputation
- domain registration age
- nameserver behaviour
- malware telemetry
- DNS response patterns
- historical threat intelligence

Therefore the appropriate interpretation is:

"DGA-like / algorithmically generated domain detection"

rather than definitive malicious-domain classification.

============================================================
END OF EXPERIMENT RECORD
============================================================
"""

output_path = "/content/experiment_results.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(results_text.strip() + "\n")

print("=" * 60)
print("✅ experiment_results.txt UPDATED")
print("=" * 60)
print(f"Path: {output_path}")
print(f"Size: {Path(output_path).stat().st_size / 1024:.2f} KB")
print("\n🔒 No dataset or model files were modified.")

✅ experiment_results.txt UPDATED
Path: /content/experiment_results.txt
Size: 10.89 KB

🔒 No dataset or model files were modified.
